[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab09_rag_and_agents.ipynb)

# Lab 9: RAG และเอเจนต์ปัญญาประดิษฐ์ (AI Agent) สำหรับงานสุขภาพ

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันเสาร์ที่ 17 ตุลาคม 2026 (ภาคบ่าย) · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- สร้างระบบ RAG (Retrieval-Augmented Generation) บนคลังความรู้ทางการแพทย์ขนาดเล็ก
- ใช้ embedding แบบโอเพนซอร์ส (open source) และฐานข้อมูลเวกเตอร์ (vector store) ซึ่งทำงานบน Colab ได้โดยไม่มีค่าใช้จ่าย
- ให้แบบจำลองภาษาขนาดใหญ่ (large language model: LLM) ตอบคำถามโดยอ้างอิงเอกสารที่ค้นคืนได้ (grounding) เพื่อลดการสร้างข้อมูลที่ไม่มีอยู่จริง (hallucination)
- กำหนดให้ระบบตอบว่า *"Not in the guidelines"* (ไม่มีในแนวทางเวชปฏิบัติ) แทนการคาดเดา
- อธิบายการพัฒนาจาก RAG ไปสู่เอเจนต์ปัญญาประดิษฐ์ที่เรียกใช้เครื่องมือ (tool) ได้

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ หรือกด **Shift+Enter**) ⭐ **Exercise** แต่ละข้อกำหนดให้เปลี่ยนค่าหรือเติมช่องว่าง (`...`) เพียงจุดเดียว ถัดจากแบบฝึกหัดทุกข้อมีเซลล์ ✅ **เฉลย** ผู้เรียนควรทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วดำเนินการในขั้นถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความ ซึ่งถือเป็น**ชิ้นงานที่ต้องส่ง** หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และ*ไม่นับ*เป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง

> **Runtime (แนะนำให้ใช้ GPU):** แบบจำลองภาษาทำงานบน GPU ได้เร็วกว่ามาก ให้เลือก `Runtime → Change runtime type → T4 GPU → Save` ใน Colab *ก่อน*รันเซลล์แรก หากไม่มี GPU ปฏิบัติการนี้ยังดำเนินการได้ แต่ LLM จะใช้เวลาตอบประมาณ 30–90 วินาทีต่อคำถาม แทนที่จะใช้เวลาเพียงไม่กี่วินาที

> ทุกขั้นตอนทำงานบน [Google Colab](https://colab.research.google.com) รุ่นที่ไม่มีค่าใช้จ่าย โดยไม่ต้องสมัครสมาชิกและไม่ต้องใช้ API key คลังความรู้ในปฏิบัติการนี้เป็น**ข้อมูลสังเคราะห์เพื่อการเรียนการสอน ไม่ใช่คำแนะนำทางคลินิก** ข้อความในคลังความรู้ คำถาม และคำสั่ง (prompt) ที่ส่งให้แบบจำลองใช้ภาษาอังกฤษ เนื่องจากแบบจำลองขนาดเล็กทำงานกับภาษาอังกฤษได้ดีที่สุด

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เป็นการ "เตรียมความพร้อม" ก่อนเริ่มปฏิบัติการ โดยทำ 3 อย่าง ได้แก่
#   1) ติดตั้งเครื่องมือ (ไลบรารี) เพิ่มเติมที่จำเป็นต้องใช้ใน Google Colab
#   2) กำหนดค่าพื้นฐาน เช่น รหัสปฏิบัติการ ชื่อที่จะแสดง และรหัสเข้าร่วมตารางอันดับ
#   3) สร้าง "ฟังก์ชัน" (ชุดคำสั่งสำเร็จรูปที่เรียกใช้ซ้ำได้) 2 ตัว คือ
#        - lab_data(...)              : ดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#        - submit_to_leaderboard(...) : ส่งผลการทำนายไปยังตารางอันดับ (ไม่บังคับ)
#   ผู้เรียนเพียงกดรันเซลล์นี้ 1 ครั้ง ไม่จำเป็นต้องแก้ไขโค้ด
#   (ยกเว้นต้องการเข้าร่วมตารางอันดับ ให้กรอก NAME และ JOIN_CODE ด้านล่าง)
# ============================================================================

# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - gradio_client   : ไลบรารีสำหรับติดต่อกับเว็บแอปตารางอันดับ (Hugging Face Space)
#   - huggingface_hub : ไลบรารีสำหรับดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face
!pip -q install gradio_client huggingface_hub

# สร้างตัวแปร (ช่องเก็บค่า) HF_OWNER แล้วใส่ข้อความ "wdittaya" ไว้
#   เครื่องหมาย = หมายถึง "นำค่าทางขวา ไปเก็บไว้ในชื่อทางซ้าย"
#   ข้อความในภาษา Python ต้องอยู่ในเครื่องหมายคำพูด " "
HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
# รหัสของปฏิบัติการนี้ ใช้ระบุว่าผลที่ส่งเป็นของปฏิบัติการใด (ไม่ต้องแก้ไข)
LAB_ID    = "lab09"
# ชื่อที่จะแสดงบนตารางอันดับ: พิมพ์ชื่อไว้ระหว่างเครื่องหมาย " " เช่น NAME = "หมอเอ"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
# รหัสเข้าร่วมตารางอันดับ: พิมพ์รหัสที่ผู้สอนแจ้งไว้ระหว่างเครื่องหมาย " "
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

# สร้างชื่อเว็บแอปตารางอันดับ โดยนำ HF_OWNER มาต่อกับข้อความ
#   - f"..." (f-string) คือข้อความที่แทรกค่าตัวแปรได้ โดยเขียนชื่อตัวแปรไว้ใน { }
#   - ผลลัพธ์คือ "wdittaya/aidh-leaderboard"
SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
# สร้างชื่อคลังข้อมูล (dataset) ของรายวิชาในลักษณะเดียวกัน ผลลัพธ์คือ "wdittaya/aidh-lab-data"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

# นำเข้า (import) ฟังก์ชัน hf_hub_download จากไลบรารี huggingface_hub
#   ฟังก์ชันนี้ใช้ดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face มาเก็บไว้ในเครื่อง
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน lab_data สำหรับดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ ตามด้วยชื่อฟังก์ชันและพารามิเตอร์ในวงเล็บ
#   - path : พารามิเตอร์ (ค่าที่ผู้ใช้ส่งเข้ามา) คือชื่อไฟล์ที่ต้องการ เช่น "pima/test_ids.csv"
#   - บรรทัดที่ย่อหน้าเข้าไปด้านล่าง คือคำสั่งที่อยู่ภายในฟังก์ชัน
# ----------------------------------------------------------------------------
def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    # เรียก hf_hub_download(...) เพื่อดาวน์โหลดไฟล์ แล้วส่งตำแหน่งไฟล์ในเครื่องกลับไป (return)
    #   - DATA_REPO           : ชื่อคลังข้อมูลบน Hugging Face ที่เก็บไฟล์
    #   - path                : ชื่อไฟล์ที่ต้องการดาวน์โหลดภายในคลังข้อมูลนั้น
    #   - repo_type="dataset" : ระบุว่าคลังนี้เป็นประเภท "ชุดข้อมูล" (ไม่ใช่โมเดล)
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน submit_to_leaderboard สำหรับส่งไฟล์ผลการทำนายไปยังตารางอันดับ
#   - path   : ชื่อไฟล์ผลการทำนายที่ต้องการส่ง
#   - lab=LAB_ID : รหัสปฏิบัติการ หากไม่ระบุ จะใช้ค่า LAB_ID ที่กำหนดไว้ด้านบนโดยอัตโนมัติ
# ----------------------------------------------------------------------------
def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    # นำเข้าเครื่องมือ 2 ตัวจากไลบรารี gradio_client
    #   - Client      : ใช้เชื่อมต่อกับเว็บแอปตารางอันดับ
    #   - handle_file : ใช้เตรียมไฟล์ในเครื่องให้อยู่ในรูปแบบที่ส่งขึ้นเว็บแอปได้
    from gradio_client import Client, handle_file
    # assert คือการตรวจสอบเงื่อนไข หากเงื่อนไขเป็นเท็จ โปรแกรมจะหยุดและแสดงข้อความเตือน
    #   - NAME and JOIN_CODE : เงื่อนไขว่าต้องกรอกทั้ง NAME และ JOIN_CODE แล้ว (ไม่เป็นข้อความว่าง)
    #   - ข้อความหลังเครื่องหมายจุลภาค คือข้อความเตือนที่จะแสดงเมื่อยังไม่ได้กรอก
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    # เชื่อมต่อกับเว็บแอปตารางอันดับ แล้วเก็บการเชื่อมต่อไว้ในตัวแปร client
    #   - SPACE_ID      : ชื่อเว็บแอปตารางอันดับที่กำหนดไว้ด้านบน
    #   - verbose=False : ไม่ต้องแสดงข้อความรายละเอียดระหว่างเชื่อมต่อ
    client = Client(SPACE_ID, verbose=False)
    # ส่งข้อมูลไปยังเว็บแอปด้วย client.predict(...) แล้วแสดงผลลัพธ์ (คะแนนและอันดับ) ด้วย print(...)
    #   - lab=lab                 : รหัสปฏิบัติการ
    #   - name=NAME               : ชื่อที่จะแสดงบนตารางอันดับ
    #   - code=JOIN_CODE          : รหัสเข้าร่วม
    #   - file=handle_file(path)  : ไฟล์ผลการทำนายที่เตรียมพร้อมสำหรับส่ง
    #   - api_name="/submit"      : ชื่อช่องทางรับข้อมูลของเว็บแอป (ส่วน "ส่งผล")
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

# แสดงข้อความยืนยันว่าการตั้งค่าเสร็จแล้ว ด้วยฟังก์ชัน print(...) (พิมพ์ข้อความออกทางหน้าจอ)
#   - ข้อความเป็น f-string จึงแทรกค่า SPACE_ID, LAB_ID และ NAME ลงในข้อความได้
#   - NAME or '(ยังไม่ได้ระบุ ...)' : หาก NAME ว่าง จะแสดงข้อความในวงเล็บแทน
print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ความจำเป็นของ RAG

ปฏิบัติการที่ 8 แสดงให้เห็นว่า LLM อาจเกิด **hallucination** คือสร้างข้อความที่ดูน่าเชื่อถือแต่ไม่มีอยู่จริง **RAG (Retrieval-Augmented Generation)** ช่วยลดปัญหานี้ได้ส่วนหนึ่ง โดยก่อนที่ LLM จะตอบ ระบบจะ*ค้นคืน* (retrieve) ข้อความที่เกี่ยวข้องที่สุดจากคลังเอกสารที่เชื่อถือได้ (ในปฏิบัติการนี้คือแนวทางเวชปฏิบัติฉบับย่อ) แล้วนำไปใส่ใน prompt คำตอบที่ได้จึง **อ้างอิงเอกสาร (grounded)** และ **ตรวจสอบแหล่งที่มาได้ (citable)** ซึ่งเป็นคุณสมบัติสำคัญสำหรับการใช้งานทางคลินิก

ระบบประกอบด้วย 3 ส่วน ซึ่งเป็นโอเพนซอร์สและไม่มีค่าใช้จ่ายทั้งหมด ได้แก่

1. **Embedding** (`sentence-transformers`): แปลงแต่ละประโยคเป็นชุดตัวเลข โดยประโยคที่มี*ความหมาย*ใกล้เคียงกันจะได้ชุดตัวเลขที่ใกล้เคียงกัน แม้จะใช้คำต่างกัน เช่น "kidney failure" กับ "low eGFR"
2. **ฐานข้อมูลเวกเตอร์ (vector store)** (`faiss`): ดัชนีสำหรับค้นหาชุดตัวเลขดังกล่าวอย่างรวดเร็ว เทียบได้กับบัตรรายการของห้องสมุดที่ค้นหาข้อความซึ่งมีความหมายใกล้เคียงกับคำถามมากที่สุด
3. **LLM** (แบบจำลอง Qwen แบบโอเพนซอร์สจากปฏิบัติการที่ 8): เรียบเรียงคำตอบจากข้อความที่ค้นคืนได้เท่านั้น

เซลล์ถัดไปจะติดตั้งเครื่องมือและดาวน์โหลดแบบจำลอง embedding ขนาดเล็ก (ใช้เวลาประมาณ 1 นาที)

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียม "เครื่องมือ" สำหรับระบบ RAG (Retrieval-Augmented Generation)
#   ซึ่งเป็นระบบที่ค้นคืนข้อความจากแนวทางเวชปฏิบัติก่อน แล้วจึงให้ LLM ตอบ โดยทำ 3 อย่าง ได้แก่
#   1) ติดตั้งไลบรารีที่จำเป็น: sentence-transformers (แปลงข้อความเป็นตัวเลข),
#      faiss (ค้นหาข้อความที่ความหมายใกล้เคียง) และ transformers (เรียกใช้ LLM)
#   2) นำเข้า (import) เครื่องมือเหล่านั้นมาใช้ในโน้ตบุ๊ก
#   3) ดาวน์โหลดแบบจำลอง embedding ขนาดเล็ก ซึ่งแปลงประโยคเป็นเวกเตอร์ (ชุดตัวเลข)
#      โดยประโยคที่ความหมายใกล้เคียงกันจะได้ชุดตัวเลขที่ใกล้เคียงกัน
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ข้อความ "แบบจำลอง embedding พร้อมใช้งาน" และสถานะว่าพบ GPU หรือไม่
#   (ใช้เวลาประมาณ 1 นาที ผู้เรียนเพียงกดรัน ไม่ต้องแก้ไขโค้ด)
# ============================================================================

# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - sentence-transformers : ไลบรารีแปลงประโยคเป็นเวกเตอร์ (embedding)
#   - faiss-cpu             : ไลบรารี FAISS (ของ Meta) สำหรับค้นหาเวกเตอร์ที่ใกล้เคียงอย่างรวดเร็ว (รุ่นที่ใช้ CPU)
#   - "transformers>=4.44"  : ไลบรารีของ Hugging Face สำหรับใช้ LLM โดยกำหนดรุ่นตั้งแต่ 4.44 ขึ้นไป
#                             (ใส่เครื่องหมายคำพูดเพื่อไม่ให้ระบบตีความเครื่องหมาย > ผิดความหมาย)
#   - accelerate            : ไลบรารีช่วยวางแบบจำลองลงบน GPU/CPU โดยอัตโนมัติ (ใช้คู่กับ device_map="auto")
!pip -q install sentence-transformers faiss-cpu "transformers>=4.44" accelerate
# นำเข้า (import) คลาส SentenceTransformer จากไลบรารี sentence_transformers
#   - from ... import ... หมายถึง "หยิบเฉพาะเครื่องมือชื่อนี้ออกมาจากไลบรารีนั้น"
#   - SentenceTransformer ใช้โหลดแบบจำลอง embedding และแปลงข้อความเป็นเวกเตอร์
from sentence_transformers import SentenceTransformer
# นำเข้าไลบรารี 3 ตัวในบรรทัดเดียว (คั่นด้วยเครื่องหมายจุลภาค ,)
#   - faiss          : ไลบรารีสร้างดัชนีค้นหาเวกเตอร์
#   - numpy as np    : ไลบรารีคำนวณตัวเลขแบบตาราง/อาร์เรย์ โดย "as np" คือการตั้งชื่อย่อว่า np
#   - torch          : ไลบรารี PyTorch สำหรับการเรียนรู้เชิงลึก ใช้ตรวจสอบ GPU และกำหนดชนิดตัวเลข
import faiss, numpy as np, torch
# นำเข้าฟังก์ชัน pipeline จากไลบรารี transformers
#   pipeline เป็นทางลัดในการสร้างระบบใช้งาน LLM ได้ในคำสั่งเดียว (จะใช้ในเซลล์โหลด LLM ภายหลัง)
from transformers import pipeline

# โหลดแบบจำลอง embedding แล้วเก็บไว้ในตัวแปร embedder
#   - ตัวแปร คือ "ช่องเก็บค่า" และเครื่องหมาย = หมายถึงนำค่าทางขวาไปเก็บไว้ในชื่อทางซ้าย
#   - SentenceTransformer(...) : สร้าง/โหลดแบบจำลองตามชื่อที่ระบุ (ดาวน์โหลดจาก Hugging Face ครั้งแรก)
#   - "all-MiniLM-L6-v2"       : ชื่อแบบจำลองขนาดเล็ก ให้เวกเตอร์ขนาด 384 มิติต่อ 1 ประโยค
#   - ข้อความหลังเครื่องหมาย # ท้ายบรรทัดเป็นคำอธิบายเดิมของผู้สอน
embedder = SentenceTransformer("all-MiniLM-L6-v2")   # แบบจำลอง embedding ขนาดเล็ก ไม่มีค่าใช้จ่าย และทำงานเร็ว
# แสดงข้อความยืนยันออกทางหน้าจอด้วย print(...)
#   - print รับค่าได้หลายค่าคั่นด้วย , และจะพิมพ์ต่อกันโดยเว้นวรรคให้
#   - torch.cuda.is_available() : ตรวจสอบว่าเครื่องนี้มี GPU (การ์ดประมวลผลกราฟิก) หรือไม่
#                                 คืนค่า True (มี) หรือ False (ไม่มี)
print("แบบจำลอง embedding พร้อมใช้งาน | ตรวจพบ GPU:", torch.cuda.is_available())

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้สร้าง "คลังความรู้" (knowledge base) ขนาดเล็ก 8 ข้อความจากแนวทางเวชปฏิบัติ
#   (ข้อมูลสังเคราะห์) แล้วทำให้ค้นหาตาม "ความหมาย" ได้ (semantic search) โดย
#   1) แปลงแต่ละข้อความเป็นเวกเตอร์ (embedding)
#   2) เก็บเวกเตอร์ทั้งหมดไว้ในดัชนี FAISS ซึ่งทำหน้าที่เป็นฐานข้อมูลเวกเตอร์ (vector store)
#   3) สร้างฟังก์ชัน search(...) และ retrieve(...) สำหรับค้นคืนข้อความที่ใกล้เคียงกับคำถามมากที่สุด
#   นี่คือส่วน "R" (Retrieval = การค้นคืน) ของ RAG ซึ่งยังไม่ใช้ LLM
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: จำนวนข้อความในดัชนี และขนาด (จำนวนมิติ) ของเวกเตอร์
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: สร้างคลังความรู้ และดัชนีค้นหาเวกเตอร์
# ----------------------------------------------------------------------------
# สร้างตัวแปร kb (ย่อมาจาก knowledge base) เก็บ "ลิสต์" (list) ของข้อความ
#   - ลิสต์ คือชุดข้อมูลหลายค่าเรียงตามลำดับ เขียนไว้ใน [ ] และคั่นแต่ละค่าด้วย ,
#   - แต่ละค่าเป็นข้อความ (string) 1 ประโยคจากแนวทางเวชปฏิบัติ (ภาษาอังกฤษ)
#   - ลำดับที่ (index) ของลิสต์ใน Python เริ่มนับจาก 0 ดังนั้นข้อความแรกคือ kb[0]
# คลังความรู้ 'แนวทางเวชปฏิบัติ' ขนาดเล็ก (ข้อมูลสังเคราะห์เพื่อการเรียนการสอน ไม่ใช่คำแนะนำทางคลินิก)
kb = [
 # ข้อความที่ 0: เกณฑ์การวินิจฉัยเบาหวานชนิดที่ 2 จาก HbA1c และระดับน้ำตาลขณะอดอาหาร
 "Type 2 diabetes is diagnosed when HbA1c is 6.5% or higher, or fasting plasma glucose is 126 mg/dL or higher.",
 # ข้อความที่ 1: ยาลำดับแรก (first-line) คือ metformin
 "First-line pharmacotherapy for type 2 diabetes is metformin unless contraindicated.",
 # ข้อความที่ 2: เป้าหมายความดันโลหิตในผู้ป่วยเบาหวาน
 "Blood pressure target for most adults with diabetes is below 130/80 mmHg.",
 # ข้อความที่ 3: การใช้ยากลุ่มสแตติน (statin) ในผู้ป่วยเบาหวานอายุ 40-75 ปี
 "Statins are recommended for diabetic patients aged 40-75 to reduce cardiovascular risk.",
 # ข้อความที่ 4: ข้อห้ามใช้ metformin เมื่อ eGFR ต่ำกว่า 30
 "Metformin is contraindicated when eGFR is below 30 mL/min/1.73m2.",
 # ข้อความที่ 5: การตรวจคัดกรองจอประสาทตา
 "Retinal (eye) screening is recommended at diagnosis of type 2 diabetes and then every 1-2 years.",
 # ข้อความที่ 6: เป้าหมาย HbA1c ทั่วไป และในผู้สูงอายุเปราะบาง
 "An HbA1c target below 7% is appropriate for most non-pregnant adults with diabetes; a less strict target (below 8%) may suit frail older adults.",
 # ข้อความที่ 7: การรักษาภาวะน้ำตาลในเลือดต่ำ (hypoglycaemia) ระดับเล็กน้อย
 "Mild hypoglycaemia (glucose below 70 mg/dL) in a conscious patient is treated with 15-20 g of fast-acting glucose, rechecking glucose after 15 minutes.",
]
# แปลงข้อความทุกข้อใน kb เป็นเวกเตอร์ แล้วเก็บไว้ในตัวแปร emb
#   - embedder.encode(...) : เมธอด (คำสั่งของออบเจกต์) จาก sentence_transformers สำหรับแปลงข้อความเป็นเวกเตอร์
#   - kb                          : ลิสต์ข้อความที่ต้องการแปลง
#   - normalize_embeddings=True   : ปรับเวกเตอร์ทุกตัวให้มีความยาวเท่ากับ 1 (normalize)
#                                   ทำให้ผลคูณภายใน (inner product) มีค่าเท่ากับความคล้ายคลึงโคไซน์ (cosine similarity)
#   - ผลลัพธ์เป็นอาร์เรย์ (ตารางตัวเลข) ขนาด 8 แถว (ข้อความ) × 384 คอลัมน์ (มิติ)
emb = embedder.encode(kb, normalize_embeddings=True)   # เวกเตอร์ (ชุดตัวเลข) 1 ชุดต่อ 1 ข้อความ
# สร้างดัชนี FAISS ที่ว่างเปล่า แล้วเก็บไว้ในตัวแปร index
#   - faiss.IndexFlatIP(...) : ดัชนีแบบ "Flat" (เปรียบเทียบกับทุกเวกเตอร์ตรง ๆ ไม่ประมาณค่า)
#                              ที่วัดความใกล้เคียงด้วยผลคูณภายใน (IP = Inner Product)
#   - emb.shape[1]           : จำนวนมิติของเวกเตอร์ (.shape บอกขนาดตาราง เป็น (จำนวนแถว, จำนวนคอลัมน์)
#                              และ [1] คือหยิบค่าลำดับที่ 1 = จำนวนคอลัมน์ = 384)
index = faiss.IndexFlatIP(emb.shape[1])                  # ดัชนีสำหรับค้นหาด้วย cosine similarity
# นำเวกเตอร์ทั้งหมดใน emb ใส่ลงในดัชนี ด้วยเมธอด .add(...) ของ FAISS
#   - emb : เวกเตอร์ของทุกข้อความ (ลำดับที่ในดัชนีจะตรงกับลำดับที่ในลิสต์ kb)
index.add(emb)
# แสดงข้อความสรุปการสร้างดัชนี
#   - index.ntotal : จำนวนเวกเตอร์ที่อยู่ในดัชนีขณะนี้ (ควรเป็น 8)
#   - emb.shape[1] : จำนวนมิติของเวกเตอร์แต่ละตัว
print("สร้างดัชนีแล้ว", index.ntotal, "ข้อความ แต่ละข้อความเป็นเวกเตอร์ขนาด", emb.shape[1], "มิติ")

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน search — ค้นหาข้อความที่ใกล้เคียงคำถามมากที่สุด พร้อมคะแนน
#   - def คือคำสั่งสร้างฟังก์ชัน (ชุดคำสั่งที่เรียกใช้ซ้ำได้) ตามด้วยชื่อและพารามิเตอร์ในวงเล็บ
#   - query : คำถามที่ต้องการค้นหา (ข้อความ)
#   - k=3   : จำนวนข้อความที่ต้องการ (top-k) หากไม่ระบุ จะใช้ค่าเริ่มต้น (default) คือ 3
#   - บรรทัดที่ย่อหน้าเข้าไปด้านล่างคือคำสั่งภายในฟังก์ชัน
# ----------------------------------------------------------------------------
def search(query, k=3):
    # คืนค่าข้อความที่คล้ายคลึงที่สุด k ข้อความ ในรูปคู่ (ข้อความ, คะแนนความคล้ายคลึง) เรียงจากมากไปน้อย
    # แปลงคำถามเป็นเวกเตอร์ด้วยวิธีเดียวกับข้อความในคลังความรู้ แล้วเก็บไว้ในตัวแปร q
    #   - [query] : ใส่คำถามไว้ในลิสต์ เพราะ encode รับเป็นลิสต์และคืนผลเป็นตาราง 1 แถว (รูปแบบที่ FAISS ต้องการ)
    #   - normalize_embeddings=True : ปรับความยาวเวกเตอร์เป็น 1 เช่นเดียวกับข้อความในคลัง
    q = embedder.encode([query], normalize_embeddings=True)
    # ค้นหาในดัชนีด้วย index.search(...) ของ FAISS แล้วรับผลลัพธ์ 2 ค่าพร้อมกัน
    #   - q : เวกเตอร์ของคำถาม
    #   - k : จำนวนข้อความที่ใกล้เคียงที่สุดที่ต้องการ
    #   - scores, idx = ... คือการแยกผลลัพธ์ 2 ค่าใส่ตัวแปร 2 ตัว:
    #       scores = คะแนนความคล้ายคลึง, idx = ลำดับที่ของข้อความในคลัง (ทั้งคู่เป็นตาราง 1 แถว เรียงจากมากไปน้อย)
    scores, idx = index.search(q, k)          # คะแนนความคล้ายคลึง และลำดับที่ของข้อความ
    # สร้างลิสต์ผลลัพธ์แล้วส่งกลับ (return) ด้วย "list comprehension" (การสร้างลิสต์จากการวนซ้ำในบรรทัดเดียว)
    #   - idx[0], scores[0]   : หยิบแถวแรก (แถวเดียว) ของผลลัพธ์ เพราะค้นหาคำถามเพียง 1 ข้อ
    #   - zip(a, b)           : ฟังก์ชันของ Python ที่จับคู่สมาชิกลำดับเดียวกันจาก 2 ลิสต์
    #   - for i, s in ...     : วนทีละคู่ โดย i = ลำดับที่ของข้อความ และ s = คะแนน
    #   - kb[i]               : หยิบข้อความลำดับที่ i จากคลังความรู้
    #   - float(s)            : แปลงคะแนนเป็นเลขทศนิยมแบบมาตรฐานของ Python
    #   - (kb[i], float(s))   : ทูเพิล (tuple) คือชุดค่าที่จับคู่กันในวงเล็บ ( )
    return [(kb[i], float(s)) for i, s in zip(idx[0], scores[0])]

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ฟังก์ชัน retrieve — คืนเฉพาะตัวข้อความ (ไม่เอาคะแนน)
#   - query : คำถาม
#   - k=3   : จำนวนข้อความที่ต้องการ (ค่าเริ่มต้น 3)
# ----------------------------------------------------------------------------
def retrieve(query, k=3):
    # คืนค่าเฉพาะตัวข้อความ (ใช้ในฟังก์ชันตอบคำถามแบบ RAG ในขั้นถัดไป)
    # เรียก search(query, k) แล้ววนทีละคู่ (p = ข้อความ, s = คะแนน) และเก็บเฉพาะ p ลงในลิสต์ใหม่
    return [p for p, s in search(query, k)]

💡 **คำอธิบายผลลัพธ์** ประโยคในแนวทางเวชปฏิบัติแต่ละประโยคถูกแปลงเป็นเวกเตอร์ (**embedding**) และจัดเก็บในดัชนี FAISS ฟังก์ชัน `search(question, k)` แปลงคำถามด้วยวิธีเดียวกัน แล้วคืนค่าข้อความ **top-k** (*k* ข้อความที่ใกล้เคียงที่สุด) พร้อม **คะแนนความคล้ายคลึง (similarity score)** โดยค่าประมาณ 1.0 หมายถึงความหมายเหมือนกัน และค่าประมาณ 0 หมายถึงไม่เกี่ยวข้องกัน ขั้นตอนนี้ยังไม่ใช้ LLM เป็นเพียงการค้นหาตามความหมาย (semantic search)

### แบบฝึกหัดที่ 1 (Exercise 1): การตั้งคำถามและการตรวจสอบข้อความที่ค้นคืนได้

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: ตั้งคำถามและตรวจสอบข้อความที่ระบบค้นคืนได้
# 1) ให้แทน ... ด้วยคำถามทางคลินิก (ภาษาอังกฤษ) ในเครื่องหมายอัญประกาศ เช่น "Which diabetes drug should be avoided in kidney failure?"
# 2) ให้กำหนด TOP_K (จำนวนข้อความที่ต้องการค้นคืน) เป็นจำนวนเต็ม เช่น 3 แล้วทดลองใช้ 1 และ 5 เพื่อเปรียบเทียบ
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้ให้ผู้เรียนลอง "ถามคำถาม" กับระบบค้นคืน (retrieval) ที่สร้างไว้ในเซลล์ก่อนหน้า
#   แล้วดูว่าระบบดึงข้อความใดจากแนวทางเวชปฏิบัติขึ้นมาบ้าง เรียงตามความคล้ายคลึงจากมากไปน้อย
#   ผู้เรียนต้องเติม 2 จุด: คำถาม (MY_QUESTION) และจำนวนข้อความที่ต้องการ (TOP_K)
#   ผลลัพธ์ที่จะเห็น: รายการข้อความอันดับ #1, #2, ... พร้อมคะแนนความคล้ายคลึง (similarity score)
#   ซึ่งค่าใกล้ 1 หมายถึงความหมายใกล้เคียงคำถามมาก และค่าใกล้ 0 หมายถึงไม่เกี่ยวข้อง
#   ยังไม่มีการใช้ LLM ในขั้นนี้
# ============================================================================

# สร้างตัวแปร MY_QUESTION เพื่อเก็บคำถามของผู้เรียน
#   - ... (จุดสามจุด) คือ "ช่องว่าง" ที่ผู้เรียนต้องเติมเอง ให้แทนด้วยคำถามภาษาอังกฤษในเครื่องหมาย " "
#   - ข้อความหลัง # ท้ายบรรทัดคือคำใบ้เดิม (TODO = สิ่งที่ต้องทำ)
MY_QUESTION = ...     # TODO: คำถามในเครื่องหมาย "อัญประกาศ"
# สร้างตัวแปร TOP_K เพื่อกำหนดจำนวนข้อความที่ต้องการให้ระบบค้นคืน (top-k)
#   - ... คือช่องว่างที่ผู้เรียนต้องเติมเป็นจำนวนเต็ม (ไม่ต้องใส่เครื่องหมายคำพูด)
TOP_K = ...           # TODO: จำนวนเต็ม เช่น 3

# วนซ้ำ (for loop) ทีละผลลัพธ์ที่ได้จากการค้นหา เพื่อแสดงผลทีละบรรทัด
#   - search(MY_QUESTION, TOP_K) : ฟังก์ชันที่สร้างไว้ในเซลล์ก่อนหน้า คืนลิสต์ของคู่ (ข้อความ, คะแนน)
#   - enumerate(..., start=1)    : ฟังก์ชันของ Python ที่ใส่ "เลขลำดับ" ให้แต่ละรายการ
#       * start=1 : ให้เริ่มนับจาก 1 (ปกติเริ่มที่ 0) เพื่อให้แสดงเป็นอันดับ 1, 2, 3 ...
#   - rank, (passage, score) : ในแต่ละรอบ แยกค่าเป็น rank = อันดับ, passage = ข้อความ, score = คะแนน
#   - เครื่องหมาย : ท้ายบรรทัด และการย่อหน้าบรรทัดถัดไป หมายถึงคำสั่งที่ทำซ้ำในแต่ละรอบ
for rank, (passage, score) in enumerate(search(MY_QUESTION, TOP_K), start=1):
    # พิมพ์ผล 1 บรรทัดด้วย f-string (ข้อความที่แทรกค่าตัวแปรใน { } ได้)
    #   - {rank}       : อันดับของข้อความ
    #   - {score:.2f}  : คะแนนความคล้ายคลึง จัดรูปแบบเป็นทศนิยม 2 ตำแหน่ง (.2f = float 2 ตำแหน่ง)
    #   - {passage}    : ตัวข้อความจากแนวทางเวชปฏิบัติ
    print(f"#{rank}  ความคล้ายคลึง={score:.2f}  {passage}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 1)
#   เซลล์นี้เป็นตัวอย่างคำตอบของแบบฝึกหัดที่ 1: ถามว่า "ยาเบาหวานตัวใดควรหลีกเลี่ยงในภาวะไตวาย"
#   แล้วให้ระบบค้นคืนข้อความที่เกี่ยวข้อง 3 อันดับแรกจากแนวทางเวชปฏิบัติ
#   จุดที่น่าสังเกต: คำถามใช้คำว่า "kidney failure" แต่ข้อความเรื่อง metformin ใช้คำว่า "eGFR"
#   ระบบ embedding ยังจับคู่ได้เพราะ "ความหมาย" สัมพันธ์กัน (semantic search)
#   ผลลัพธ์ที่จะเห็น: ข้อความ 3 อันดับ พร้อมคะแนนความคล้ายคลึง
# ============================================================================

# เก็บคำถามทางคลินิก (ภาษาอังกฤษ) ไว้ในตัวแปร MY_QUESTION
#   ข้อความ (string) ต้องอยู่ในเครื่องหมายคำพูด " "
MY_QUESTION = "Which diabetes drug should be avoided in kidney failure?"
# กำหนดให้ค้นคืนข้อความ 3 อันดับแรก (top-3) โดยเป็นจำนวนเต็ม จึงไม่ต้องใส่เครื่องหมายคำพูด
TOP_K = 3

# วนซ้ำทีละผลลัพธ์จาก search(MY_QUESTION, TOP_K)
#   - search(...)             : คืนลิสต์ของคู่ (ข้อความ, คะแนน) เรียงจากคล้ายมากไปน้อย
#   - enumerate(..., start=1) : ใส่เลขลำดับให้แต่ละรายการ โดยเริ่มนับจาก 1
#   - rank, (passage, score)  : แยกเป็นอันดับ, ข้อความ และคะแนน ในแต่ละรอบ
for rank, (passage, score) in enumerate(search(MY_QUESTION, TOP_K), start=1):
    # พิมพ์ผล 1 บรรทัดด้วย f-string
    #   - {score:.2f} : แสดงคะแนนเป็นทศนิยม 2 ตำแหน่ง
    print(f"#{rank}  ความคล้ายคลึง={score:.2f}  {passage}")

💡 **คำอธิบายผลลัพธ์** คำถามที่ใช้คำว่า "kidney failure" ไม่มีคำว่า *eGFR* ร่วมกับข้อความเรื่อง metformin แต่ embedding มักจัดให้ข้อความดังกล่าวอยู่ในอันดับต้น เนื่องจาก*ความหมาย*สัมพันธ์กัน ข้อความในอันดับถัดลงไปมีคะแนนความคล้ายคลึงต่ำกว่า และจะถูกส่งให้ LLM ด้วยหากกำหนด `TOP_K` สูงขึ้น

✍️ **คำถามที่ 1** ข้อความที่เป็นประโยชน์ที่สุดอยู่ในอันดับที่ 1 หรือไม่ เมื่อเพิ่มค่า `TOP_K` คะแนนความคล้ายคลึงของข้อความในอันดับท้ายเปลี่ยนแปลงอย่างไร และจงอธิบายว่าเหตุใดการส่งข้อความให้ LLM *มากเกินไป*จึงอาจก่อให้เกิดปัญหา

✍️ **คำตอบ:**

## การโหลดแบบจำลองภาษา (LLM)

ขั้นตอนนี้โหลด LLM ขนาดเล็กแบบโอเพนซอร์สตัวเดียวกับปฏิบัติการที่ 8 (Qwen2.5-1.5B-Instruct ขนาดประมาณ 3 GB ใช้เวลาดาวน์โหลด 1–2 นาที) ฟังก์ชัน `generate(system, user)` ส่งคำสั่ง **system** (ข้อกำหนดที่แบบจำลองต้องปฏิบัติตาม) และข้อความ **user** (คำถาม) ไปยังแบบจำลอง แล้วคืนค่าคำตอบของแบบจำลอง

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้โหลดแบบจำลองภาษาขนาดใหญ่ (large language model: LLM) ชื่อ Qwen2.5-1.5B-Instruct
#   (ตัวเดียวกับปฏิบัติการที่ 8 ขนาดประมาณ 3 GB) ซึ่งเป็นส่วน "G" (Generation = การสร้างคำตอบ) ของ RAG
#   และสร้างฟังก์ชัน generate(system, user) สำหรับส่งข้อความไปถามแบบจำลอง โดยมี 2 ส่วน คือ
#     - system : ข้อกำหนด/บทบาทที่แบบจำลองต้องปฏิบัติตาม
#     - user   : คำถามหรือข้อความจากผู้ใช้
#   ผลลัพธ์ที่ผู้เรียนจะเห็น: ข้อความ "LLM พร้อมใช้งาน" (ใช้เวลาดาวน์โหลด 1–2 นาที)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: โหลด LLM
# ----------------------------------------------------------------------------
# สร้างระบบใช้งาน LLM ด้วย pipeline(...) จากไลบรารี transformers แล้วเก็บไว้ในตัวแปร llm
#   (คำสั่งนี้ยาวหลายบรรทัด เพราะอยู่ในวงเล็บ ( ) Python จึงอ่านต่อกันเป็นคำสั่งเดียว)
#   - "text-generation"                       : ประเภทงาน คือ "สร้างข้อความต่อ" (ให้แบบจำลองเขียนคำตอบ)
#   - model="Qwen/Qwen2.5-1.5B-Instruct"      : ชื่อแบบจำลองบน Hugging Face (1.5 พันล้านพารามิเตอร์
#                                               รุ่น Instruct ที่ฝึกให้ทำตามคำสั่ง)
#   - torch_dtype=...                         : ชนิดตัวเลขที่ใช้เก็บค่าในแบบจำลอง
#       * torch.float16 if torch.cuda.is_available() else torch.float32
#         คือการเลือกแบบมีเงื่อนไขในบรรทัดเดียว: ถ้ามี GPU ใช้ float16 (ใช้หน่วยความจำครึ่งหนึ่ง เร็วกว่า)
#         ถ้าไม่มี GPU ใช้ float32 (ความละเอียดมาตรฐานที่ CPU รองรับได้ดี)
#   - device_map="auto"                       : ให้ระบบ (ไลบรารี accelerate) เลือกวางแบบจำลองบน GPU หรือ CPU เองโดยอัตโนมัติ
# LLM (แบบจำลองโอเพนซอร์สตัวเดียวกับปฏิบัติการที่ 8) ใช้ float16 บน GPU และ float32 บน CPU
llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน generate — ส่งคำสั่ง system และข้อความ user ให้ LLM แล้วคืนคำตอบ
#   - system             : ข้อกำหนดที่แบบจำลองต้องปฏิบัติตาม (ข้อความ)
#   - user               : คำถาม/ข้อความของผู้ใช้ (ข้อความ)
#   - max_new_tokens=250 : จำนวนโทเคน (token = หน่วยย่อยของคำ) สูงสุดที่ให้แบบจำลองเขียนได้ ค่าเริ่มต้น 250
# ----------------------------------------------------------------------------
def generate(system, user, max_new_tokens=250):
    # สร้างตัวแปร msg เก็บ "บทสนทนา" ในรูปแบบที่ LLM แบบแชตต้องการ
    #   - เป็นลิสต์ [ ] ที่มีสมาชิก 2 ตัว แต่ละตัวเป็นดิกชันนารี (dict) { }
    #   - dict คือชุดข้อมูลแบบ "กุญแจ: ค่า" (key: value) เช่น "role": "system"
    #   - "role"    : บทบาทของผู้พูด ได้แก่ "system" (ข้อกำหนด) และ "user" (ผู้ใช้)
    #   - "content" : เนื้อหาข้อความของบทบาทนั้น (มาจากพารามิเตอร์ system และ user)
    msg = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    # ส่งบทสนทนาให้ LLM สร้างคำตอบ แล้วดึงเฉพาะข้อความคำตอบส่งกลับ (return) โดยทำทีละขั้นดังนี้
    #   1) llm(msg, ...) : เรียกใช้แบบจำลอง
    #        - msg                           : บทสนทนาที่เตรียมไว้
    #        - max_new_tokens=max_new_tokens : ความยาวคำตอบสูงสุด (ส่งต่อค่าจากพารามิเตอร์ของฟังก์ชัน)
    #        - do_sample=False               : ไม่สุ่ม ให้เลือกคำที่น่าจะเป็นที่สุดทุกครั้ง
    #                                          (greedy decoding) จึงได้คำตอบเดิมทุกครั้งที่รัน
    #   2) [0] : ผลลัพธ์เป็นลิสต์ หยิบผลลัพธ์ชุดแรก (ลำดับที่ 0)
    #   3) ["generated_text"] : หยิบค่าของกุญแจ "generated_text" ซึ่งคือบทสนทนาทั้งหมด (รวมคำตอบใหม่ต่อท้าย)
    #   4) [-1] : หยิบข้อความ "สุดท้าย" ของบทสนทนา (ดัชนี -1 = ตัวท้ายสุด) ซึ่งคือคำตอบของแบบจำลอง
    #   5) ["content"] : หยิบเฉพาะเนื้อหาข้อความของคำตอบนั้น
    return llm(msg, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"][-1]["content"]

# แสดงข้อความยืนยันว่าโหลด LLM เสร็จแล้ว
print("LLM พร้อมใช้งาน")

### แบบฝึกหัดที่ 2 (Exercise 2): prompt สำหรับ RAG

ฟังก์ชัน `answer_with_rag()` ค้นคืนข้อความที่เกี่ยวข้อง กำกับหมายเลข [1], [2], ... เป็น "บริบท (context)" แล้วสั่งให้ LLM ตอบ*จากบริบทเท่านั้น* พร้อมอ้างอิงหมายเลขข้อความ หรือตอบว่า `Not in the guidelines.` หากบริบทไม่มีคำตอบ คำสั่งที่ส่งให้แบบจำลองเขียนเป็นภาษาอังกฤษ เนื่องจากแบบจำลองขนาดเล็กปฏิบัติตามคำสั่งภาษาอังกฤษได้แม่นยำกว่า และต้องคงวลี `"Not in the guidelines."` ไว้ตามเดิม เนื่องจากระบบให้คะแนนใช้วลีนี้ในการตรวจสอบ

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: prompt สำหรับ RAG
# ข้อความ user ที่ส่งให้แบบจำลองต้องประกอบด้วยทั้ง "บริบท" และ "คำถาม" โดยส่วนบริบทได้เตรียมไว้แล้ว
# ให้แทน ... ด้วยชื่อตัวแปรที่เก็บคำถาม   คำใบ้: ชื่อตัวแปรคือ  question
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้รวม "การค้นคืน" (retrieval) กับ "LLM" เข้าด้วยกันเป็นระบบ RAG ที่สมบูรณ์
#   ผ่านฟังก์ชัน answer_with_rag(question) ซึ่งทำงาน 3 ขั้น:
#     1) ค้นคืนข้อความที่เกี่ยวข้องกับคำถามจากแนวทางเวชปฏิบัติ
#     2) ใส่หมายเลข [1], [2], ... ให้แต่ละข้อความ แล้วรวมเป็น "บริบท" (context)
#     3) สั่ง LLM ให้ตอบ "จากบริบทเท่านั้น" พร้อมอ้างอิงหมายเลขข้อความ (citation)
#        หรือตอบว่า "Not in the guidelines." หากบริบทไม่มีคำตอบ (การงดตอบ = abstention)
#   ผู้เรียนต้องเติมช่องว่าง ... เพียง 1 จุด เพื่อให้ข้อความที่ส่งให้ LLM มีคำถามรวมอยู่ด้วย
#   ผลลัพธ์ที่จะเห็น: บริบทที่ค้นคืนได้ และคำตอบของ LLM สำหรับคำถามเรื่อง eGFR 25 กับ metformin
# ============================================================================

# กำหนดวลีมาตรฐานสำหรับ "งดตอบ" เก็บไว้ในตัวแปร ABSTAIN_PHRASE
#   ต้องคงวลีนี้ไว้ตามเดิม เพราะระบบให้คะแนนใช้วลีนี้ตรวจสอบว่าระบบงดตอบหรือไม่
ABSTAIN_PHRASE = "Not in the guidelines."

# ----------------------------------------------------------------------------
# ฟังก์ชัน answer_with_rag — ตอบคำถามโดยอ้างอิงแนวทางเวชปฏิบัติที่ค้นคืนได้
#   - question           : คำถามของผู้ใช้ (ข้อความ)
#   - k=3                : จำนวนข้อความที่ค้นคืนมาใช้เป็นบริบท (ค่าเริ่มต้น 3)
#   - show_context=False : ถ้าเป็น True จะพิมพ์บริบทที่ค้นคืนได้ให้ดูด้วย (ค่าเริ่มต้นคือไม่พิมพ์)
# ----------------------------------------------------------------------------
def answer_with_rag(question, k=3, show_context=False):
    # ขั้นที่ 1: ค้นคืนข้อความ k ข้อความที่ใกล้เคียงคำถามที่สุด ด้วยฟังก์ชัน retrieve (สร้างไว้ก่อนหน้า)
    #   ผลลัพธ์เป็นลิสต์ของข้อความ เก็บไว้ในตัวแปร passages
    passages = retrieve(question, k)
    # ขั้นที่ 2: รวมข้อความทั้งหมดเป็นบริบทข้อความเดียว โดยใส่หมายเลขกำกับ
    #   - enumerate(passages) : ให้ลำดับ i (เริ่มที่ 0) คู่กับข้อความ c ในแต่ละรอบ
    #   - f"[{i+1}] {c}"      : สร้างข้อความรูปแบบ "[1] ข้อความ..." (บวก 1 เพื่อให้เลขเริ่มที่ 1)
    #   - "\n".join(...)      : เมธอดของข้อความ นำทุกบรรทัดมาต่อกัน โดยคั่นด้วย "\n" (ขึ้นบรรทัดใหม่)
    context = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(passages))
    # if คือการตรวจเงื่อนไข: ถ้า show_context เป็น True จึงทำคำสั่งที่ย่อหน้าอยู่ด้านล่าง
    if show_context:
        # พิมพ์บริบทที่ค้นคืนได้ให้ผู้เรียนดู
        #   - เครื่องหมาย + ใช้ต่อข้อความเข้าด้วยกัน
        #   - "\n" คือการขึ้นบรรทัดใหม่
        print("บริบทที่ค้นคืนได้:\n" + context + "\n")
    # สร้างคำสั่ง system (ข้อกำหนดให้ LLM) เป็นภาษาอังกฤษ เก็บไว้ในตัวแปร system
    #   - ข้อความหลายชิ้นที่วางติดกันภายในวงเล็บ ( ) Python จะนำมาต่อกันเป็นข้อความเดียวโดยอัตโนมัติ
    #   - ใจความ: "คุณเป็นผู้ช่วยด้านแนวทางเวชปฏิบัติ ให้ตอบโดยใช้บริบทที่ให้มาเท่านั้น
    #             หากบริบทไม่มีคำตอบ ให้ตอบตรงตามวลี ABSTAIN_PHRASE และอ้างอิงหมายเลขข้อความ เช่น [1]"
    #   - ชิ้นสุดท้ายเป็น f-string เพื่อแทรกค่า ABSTAIN_PHRASE ลงในคำสั่ง
    #   - ใช้ภาษาอังกฤษเพราะแบบจำลองขนาดเล็กปฏิบัติตามคำสั่งภาษาอังกฤษได้แม่นยำกว่า
    system = ("You are a clinical guideline assistant. Answer ONLY using the "
              "provided context. If the answer is not in the context, reply exactly "
              f"'{ABSTAIN_PHRASE}' Cite passage numbers like [1].")
    # ขั้นที่ 3: สร้างข้อความ user ที่จะส่งให้ LLM ประกอบด้วยบริบทและคำถาม
    #   - f"Context:\n{context}\n\nQuestion: " : ส่วนบริบทที่เตรียมไว้แล้ว ตามด้วยหัวข้อ "Question: "
    #   - + ...  : ต่อท้ายด้วยสิ่งที่ผู้เรียนต้องเติม (... คือช่องว่างของแบบฝึกหัด ให้ดูคำใบ้ด้านบนสุดของเซลล์)
    user = f"Context:\n{context}\n\nQuestion: " + ...   # TODO: เพิ่มคำถาม
    # ส่งคำสั่ง system และข้อความ user ให้ LLM ผ่านฟังก์ชัน generate แล้วคืนคำตอบที่ได้
    return generate(system, user)

# ทดลองเรียกใช้ฟังก์ชันกับคำถาม "ผู้ป่วยที่มี eGFR 25 ใช้ metformin ได้หรือไม่"
#   - show_context=True : ให้พิมพ์บริบทที่ค้นคืนได้ก่อนแสดงคำตอบ
#   - print(...)        : แสดงคำตอบของ LLM ออกทางหน้าจอ
print(answer_with_rag("Can a patient with eGFR 25 take metformin?", show_context=True))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 2)
#   เซลล์นี้สร้างฟังก์ชัน answer_with_rag(question) ที่สมบูรณ์ ซึ่งเป็นหัวใจของระบบ RAG:
#     ค้นคืนข้อความ → ใส่หมายเลขเป็นบริบท → สั่ง LLM ให้ตอบจากบริบทเท่านั้นพร้อมอ้างอิง [n]
#     หรือตอบว่า "Not in the guidelines." หากบริบทไม่มีคำตอบ (การงดตอบ = abstention)
#   คำตอบของแบบฝึกหัด: ต่อท้ายข้อความ user ด้วยตัวแปร question (คำถามของผู้ใช้)
#   ผลลัพธ์ที่จะเห็น: บริบทที่ค้นคืนได้ และคำตอบของ LLM ที่อ้างอิงหมายเลขข้อความ
# ============================================================================

# กำหนดวลีมาตรฐานสำหรับการงดตอบ (ต้องคงไว้ตามเดิม เพราะระบบให้คะแนนใช้วลีนี้ตรวจสอบ)
ABSTAIN_PHRASE = "Not in the guidelines."

# ----------------------------------------------------------------------------
# ฟังก์ชัน answer_with_rag — ตอบคำถามโดยอ้างอิงแนวทางเวชปฏิบัติที่ค้นคืนได้
#   - question           : คำถามของผู้ใช้
#   - k=3                : จำนวนข้อความที่ค้นคืนมาเป็นบริบท (ค่าเริ่มต้น 3)
#   - show_context=False : ถ้าเป็น True จะพิมพ์บริบทให้ดูด้วย
# ----------------------------------------------------------------------------
def answer_with_rag(question, k=3, show_context=False):
    # ค้นคืนข้อความ k ข้อความที่ใกล้เคียงคำถามที่สุด (ลิสต์ของข้อความ)
    passages = retrieve(question, k)
    # รวมข้อความเป็นบริบทเดียว โดยแต่ละบรรทัดมีรูปแบบ "[หมายเลข] ข้อความ"
    #   - enumerate(passages) : ให้ลำดับ i (เริ่ม 0) คู่กับข้อความ c
    #   - f"[{i+1}] {c}"      : ใส่หมายเลขเริ่มจาก 1
    #   - "\n".join(...)      : ต่อทุกบรรทัดเข้าด้วยกัน คั่นด้วยการขึ้นบรรทัดใหม่
    context = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(passages))
    # ถ้า show_context เป็น True ให้พิมพ์บริบทออกมาดู
    if show_context:
        # พิมพ์บริบท (เครื่องหมาย + ใช้ต่อข้อความ, "\n" คือขึ้นบรรทัดใหม่)
        print("บริบทที่ค้นคืนได้:\n" + context + "\n")
    # สร้างคำสั่ง system ภาษาอังกฤษ (ข้อความหลายชิ้นในวงเล็บถูกต่อกันเป็นข้อความเดียวโดยอัตโนมัติ)
    #   ใจความ: ให้ตอบจากบริบทเท่านั้น หากไม่มีคำตอบให้ตอบตรงตามวลี ABSTAIN_PHRASE และอ้างอิงหมายเลขข้อความ เช่น [1]
    #   ชิ้นสุดท้ายเป็น f-string เพื่อแทรกค่า ABSTAIN_PHRASE
    system = ("You are a clinical guideline assistant. Answer ONLY using the "
              "provided context. If the answer is not in the context, reply exactly "
              f"'{ABSTAIN_PHRASE}' Cite passage numbers like [1].")
    # สร้างข้อความ user = บริบท + หัวข้อ "Question: " + คำถามของผู้ใช้ (ตัวแปร question)
    #   นี่คือคำตอบของแบบฝึกหัด: ต้องต่อท้ายด้วย question เพื่อให้ LLM รู้ว่าถูกถามอะไร
    user = f"Context:\n{context}\n\nQuestion: " + question
    # ส่ง system และ user ให้ LLM ผ่านฟังก์ชัน generate แล้วคืนคำตอบ
    return generate(system, user)

# ทดลองถามว่า "ผู้ป่วยที่มี eGFR 25 ใช้ metformin ได้หรือไม่" พร้อมแสดงบริบท (show_context=True)
#   แล้วพิมพ์คำตอบของ LLM ด้วย print(...)
print(answer_with_rag("Can a patient with eGFR 25 take metformin?", show_context=True))

💡 **คำอธิบายผลลัพธ์** ขั้นตอนนี้คือหลักการสำคัญของ RAG ได้แก่ *ค้นคืนข้อความ → นำข้อความใส่ใน prompt → สั่งให้ LLM ตอบจากข้อความดังกล่าวเท่านั้นพร้อมอ้างอิงแหล่งที่มา* นอกจากนี้ คำสั่ง system ยังกำหนดทางเลือกที่ปลอดภัยให้แบบจำลอง ("Not in the guidelines.") ซึ่งเรียกว่า **การงดตอบ (abstention)**

### แบบฝึกหัดที่ 3 (Exercise 3): การเปรียบเทียบคำตอบที่ไม่ใช้และใช้ RAG

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: เปรียบเทียบคำตอบที่ไม่ค้นคืนเอกสารกับคำตอบที่ค้นคืนเอกสาร และประเมินคำตอบ
# ให้เลือกคำถาม 1 ข้อ โดยแทน ... ด้วยคำถามข้อใดข้อหนึ่งด้านล่าง (คัดลอกรวมเครื่องหมายอัญประกาศ) หรือตั้งคำถามเอง
#   "At what HbA1c is type 2 diabetes diagnosed?"
#   "What HbA1c target suits a frail 85-year-old with diabetes?"
#   "How should mild hypoglycaemia be treated in a conscious patient?"
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้ถามคำถามเดียวกันกับ LLM 2 แบบ เพื่อให้ผู้เรียนเปรียบเทียบ
#     A) ไม่ใช้ RAG : LLM ตอบจากความรู้ที่เรียนรู้มาระหว่างการฝึกเท่านั้น (ตรวจสอบแหล่งที่มาไม่ได้)
#     B) ใช้ RAG    : LLM ตอบจากข้อความแนวทางเวชปฏิบัติที่ค้นคืนได้ พร้อมอ้างอิงหมายเลข [n]
#   ผู้เรียนต้องเติมคำถาม 1 ข้อแทน ... แล้วนำผลไปประเมินในตารางของคำถามที่ 2
#   ผลลัพธ์ที่จะเห็น: คำถาม, คำตอบ A, บริบทที่ค้นคืนได้ และคำตอบ B
# ============================================================================

# สร้างตัวแปร COMPARE_QUESTION เก็บคำถามที่จะใช้เปรียบเทียบ
#   - ... คือช่องว่างที่ผู้เรียนต้องเติมเป็นคำถาม (ข้อความในเครื่องหมาย " ") ตามรายการด้านบนหรือคำถามของตนเอง
COMPARE_QUESTION = ...   # TODO

# แสดงคำถามที่เลือก (print รับหลายค่าคั่นด้วย , และพิมพ์ต่อกันโดยเว้นวรรค)
print("คำถาม:", COMPARE_QUESTION)
# แบบ A: ถาม LLM โดยตรง "ไม่ใช้ RAG" แล้วพิมพ์หัวข้อพร้อมคำตอบ
#   - generate(system, user) : ฟังก์ชันส่งข้อความให้ LLM (สร้างไว้ในเซลล์โหลด LLM)
#       * "You are a clinical assistant." : คำสั่ง system ทั่วไป ("คุณเป็นผู้ช่วยทางคลินิก") ไม่มีบริบทใด ๆ
#       * COMPARE_QUESTION                : คำถามของผู้เรียน
#   - "\n" คือขึ้นบรรทัดใหม่ และเครื่องหมาย + ใช้ต่อหัวข้อเข้ากับคำตอบ
print("\n--- A) ไม่ใช้ RAG (อาศัยความรู้ที่แบบจำลองเรียนรู้มา) ---\n" + generate("You are a clinical assistant.", COMPARE_QUESTION))
# พิมพ์หัวข้อของแบบ B
print("\n--- B) ใช้ RAG (อ้างอิงแนวทางเวชปฏิบัติ) ---")
# แบบ B: ถามด้วยระบบ RAG (ฟังก์ชัน answer_with_rag จากแบบฝึกหัดที่ 2) แล้วพิมพ์คำตอบ
#   - COMPARE_QUESTION   : คำถามเดียวกับแบบ A
#   - show_context=True  : แสดงข้อความที่ค้นคืนได้ เพื่อให้ตรวจสอบว่าคำตอบมีหลักฐานรองรับหรือไม่
print(answer_with_rag(COMPARE_QUESTION, show_context=True))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 3)
#   เซลล์นี้ใช้คำถาม "HbA1c ระดับใดใช้วินิจฉัยเบาหวานชนิดที่ 2" แล้วถาม LLM 2 แบบเพื่อเปรียบเทียบ
#     A) ไม่ใช้ RAG : ตอบจากความรู้เดิมของแบบจำลอง (ไม่มีแหล่งอ้างอิง)
#     B) ใช้ RAG    : ตอบจากแนวทางเวชปฏิบัติที่ค้นคืนได้ พร้อมอ้างอิงหมายเลข [n]
#   ผลลัพธ์ที่จะเห็น: คำถาม, คำตอบ A, บริบทที่ค้นคืนได้ และคำตอบ B สำหรับนำไปประเมินในคำถามที่ 2
# ============================================================================

# เก็บคำถามที่ใช้เปรียบเทียบไว้ในตัวแปร COMPARE_QUESTION (ข้อความภาษาอังกฤษในเครื่องหมาย " ")
COMPARE_QUESTION = "At what HbA1c is type 2 diabetes diagnosed?"

# แสดงคำถามที่ใช้
print("คำถาม:", COMPARE_QUESTION)
# แบบ A: เรียก generate(...) ด้วยคำสั่ง system ทั่วไป "You are a clinical assistant." และคำถามโดยตรง (ไม่มีบริบท)
#   แล้วนำหัวข้อมาต่อกับคำตอบด้วยเครื่องหมาย + ก่อนพิมพ์ ("\n" = ขึ้นบรรทัดใหม่)
print("\n--- A) ไม่ใช้ RAG (อาศัยความรู้ที่แบบจำลองเรียนรู้มา) ---\n" + generate("You are a clinical assistant.", COMPARE_QUESTION))
# พิมพ์หัวข้อของแบบ B
print("\n--- B) ใช้ RAG (อ้างอิงแนวทางเวชปฏิบัติ) ---")
# แบบ B: ถามด้วย answer_with_rag(...) โดย show_context=True เพื่อแสดงข้อความที่ค้นคืนได้ แล้วพิมพ์คำตอบ
print(answer_with_rag(COMPARE_QUESTION, show_context=True))

💡 **คำอธิบายผลลัพธ์** คำตอบ A มาจากความรู้ที่แบบจำลองเรียนรู้ระหว่างการฝึกเท่านั้น จึงอาจถูกต้อง ล้าสมัย หรือไม่มีอยู่จริง และไม่สามารถตรวจสอบแหล่งที่มาได้ ส่วนคำตอบ B ควรอิงตามข้อความที่ค้นคืนได้และอ้างอิงหมายเลขข้อความ ซึ่งช่วยให้บุคลากรทางการแพทย์ตรวจสอบได้ภายในเวลาอันสั้น

✍️ **คำถามที่ 2 — การประเมินคำตอบในฐานะผู้ตรวจสอบทางคลินิก** ให้คัดลอกตารางต่อไปนี้ไปไว้ในเซลล์คำตอบและกรอกผล (ใช่ / ไม่ใช่ / บางส่วน)

| | ถูกต้องหรือไม่ | มีข้อความที่ค้นคืนได้รองรับหรือไม่ | อ้างอิงหมายเลขข้อความ [n] หรือไม่ | มีข้อมูลที่ไม่มีหลักฐานรองรับหรือไม่ปลอดภัยหรือไม่ |
|---|---|---|---|---|
| A) ไม่ใช้ RAG | | | | |
| B) ใช้ RAG | | | | |

จากผลลัพธ์ข้างต้น ผู้เรียนพิจารณาว่าคำตอบแบบใดเหมาะสมที่จะนำเสนอต่อแพทย์ผู้มีภาระงานมาก เพราะเหตุใด

✍️ **คำตอบ:**

## คำถามนอกขอบเขตและเกณฑ์การงดตอบ (abstention threshold)

ระบบผู้ช่วยทางคลินิกที่ปลอดภัยต้อง**งดตอบ**เมื่อแนวทางเวชปฏิบัติไม่ครอบคลุมคำถาม แทนการคาดเดา วิธีหนึ่งคือการใช้กฎอย่างง่าย*ก่อน*ส่งคำถามให้ LLM กล่าวคือ หากข้อความที่ค้นคืนได้ในอันดับที่ 1 มี **คะแนนความคล้ายคลึงต่ำกว่าเกณฑ์ตัดสิน (threshold)** ให้ตอบ "Not in the guidelines." ทันที

เซลล์ต่อไปนี้แสดงคะแนนความคล้ายคลึงสูงสุดของชุดคำถามฝึกหัด ซึ่งมีทั้งคำถามที่แนวทางเวชปฏิบัติฉบับย่อครอบคลุม (**อยู่ในขอบเขต / in scope**) และคำถามที่ไม่ครอบคลุม (**นอกขอบเขต / out of scope**) ขั้นตอนนี้ไม่ใช้ LLM จึงทำงานได้รวดเร็ว

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียมข้อมูลสำหรับกำหนด "เกณฑ์การงดตอบ" (abstention threshold) โดย
#   1) สร้างชุดคำถามฝึกหัด 8 ข้อ ซึ่งมีทั้งคำถามที่แนวทางเวชปฏิบัติฉบับย่อครอบคลุม
#      (อยู่ในขอบเขต / in scope) และคำถามที่ไม่ครอบคลุม (นอกขอบเขต / out of scope)
#   2) สร้างฟังก์ชัน best_score(question) ที่คืนคะแนนความคล้ายคลึงของข้อความอันดับที่ 1
#   3) แสดงตารางคะแนนสูงสุดของคำถามแต่ละข้อ
#   แนวคิด: คำถามในขอบเขตมักได้คะแนนสูง ส่วนคำถามนอกขอบเขตมักได้คะแนนต่ำ
#   จึงใช้คะแนนนี้คัดกรองคำถามก่อนส่งให้ LLM ได้ (ขั้นตอนนี้ไม่ใช้ LLM จึงทำงานเร็ว)
#   ผลลัพธ์ที่จะเห็น: ตาราง 3 คอลัมน์ ได้แก่ คะแนนสูงสุด, อยู่ในขอบเขตหรือไม่, และคำถาม
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: ชุดคำถามฝึกหัด
# ----------------------------------------------------------------------------
# สร้างตัวแปร practice เป็นลิสต์ของทูเพิล (tuple) แต่ละทูเพิลมี 2 ค่าในวงเล็บ ( ) ได้แก่
#   - ค่าแรก  : คำถาม (ข้อความภาษาอังกฤษ)
#   - ค่าที่สอง : True (จริง) = อยู่ในขอบเขตของแนวทางเวชปฏิบัติฉบับย่อ, False (เท็จ) = นอกขอบเขต
#   (True/False เป็นค่าตรรกะ หรือ boolean ของ Python)
practice = [   # (คำถาม, แนวทางเวชปฏิบัติฉบับย่อครอบคลุมหรือไม่)
    # ในขอบเขต: เกณฑ์ HbA1c สำหรับวินิจฉัยเบาหวานชนิดที่ 2
    ("At what HbA1c level is type 2 diabetes diagnosed?",               True),
    # ในขอบเขต: ผู้ป่วย eGFR 25 ใช้ metformin ต่อได้หรือไม่
    ("Can a patient with an eGFR of 25 keep taking metformin?",         True),
    # ในขอบเขต: ความถี่ในการตรวจตาของผู้ป่วยเบาหวาน
    ("How often should people with diabetes have their eyes checked?",  True),
    # ในขอบเขต: เป้าหมายความดันโลหิตของผู้ป่วยเบาหวาน
    ("What blood pressure should a diabetic patient aim for?",          True),
    # นอกขอบเขต: การรักษาเหาในเด็ก
    ("What is the first-line treatment for head lice in children?",     False),
    # นอกขอบเขต: การดูแลข้อเท้าแพลงใน 48 ชั่วโมงแรก
    ("How should a sprained ankle be managed in the first 48 hours?",  False),
    # นอกขอบเขต: วัคซีนสำหรับทารกอายุ 2 เดือน
    ("Which vaccines are due at a baby's 2-month visit?",              False),
    # นอกขอบเขต: ยาต้านซึมเศร้าลำดับแรกสำหรับโรคซึมเศร้า
    ("What is the first-line antidepressant for major depression?",     False),
]

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ฟังก์ชัน best_score — คะแนนความคล้ายคลึงสูงสุดของคำถาม
#   - question : คำถามที่ต้องการตรวจสอบ
# ----------------------------------------------------------------------------
def best_score(question):
    # ค้นหาข้อความที่ใกล้เคียงที่สุดเพียง 1 ข้อความ แล้วคืนเฉพาะคะแนน โดยทำทีละขั้น
    #   1) search(question, 1) : ค้นหา top-1 ได้ลิสต์ที่มีคู่ (ข้อความ, คะแนน) 1 คู่
    #   2) [0]                 : หยิบคู่แรก (อันดับที่ 1)
    #   3) [1]                 : หยิบค่าลำดับที่ 1 ของคู่นั้น คือ "คะแนน" (ลำดับที่ 0 คือข้อความ)
    return search(question, 1)[0][1]      # คะแนนความคล้ายคลึงของข้อความอันดับที่ 1

# ----------------------------------------------------------------------------
# ส่วนที่ 3: แสดงตารางคะแนน
# ----------------------------------------------------------------------------
# พิมพ์หัวตารางด้วย f-string ที่มีการจัดรูปแบบความกว้าง
#   - {'คะแนนสูงสุด':>10} : แสดงข้อความ "คะแนนสูงสุด" ในช่องกว้าง 10 ตัวอักษร ชิดขวา (> = ชิดขวา)
#   - {'ในขอบเขต':9}      : แสดงข้อความ "ในขอบเขต" ในช่องกว้าง 9 ตัวอักษร (ข้อความชิดซ้ายโดยปริยาย)
#   (การจัดความกว้างช่วยให้คอลัมน์ตรงกัน แต่ภาษาไทยที่มีสระบน/ล่างอาจทำให้ไม่ตรงพอดี)
print(f"{'คะแนนสูงสุด':>10}  {'ในขอบเขต':9}  คำถาม")
# วนซ้ำทีละคำถามในชุดฝึกหัด
#   - qtext    : ข้อความคำถาม (ค่าแรกของทูเพิล)
#   - in_scope : True/False ว่าอยู่ในขอบเขตหรือไม่ (ค่าที่สองของทูเพิล)
for qtext, in_scope in practice:
    # พิมพ์ 1 แถวของตาราง
    #   - {best_score(qtext):10.2f} : คำนวณคะแนนสูงสุดของคำถาม แล้วแสดงในช่องกว้าง 10 ทศนิยม 2 ตำแหน่ง
    #   - {'ใช่' if in_scope else 'ไม่ใช่':9} : เลือกข้อความแบบมีเงื่อนไขในบรรทัดเดียว
    #       ถ้า in_scope เป็น True แสดง "ใช่" มิฉะนั้นแสดง "ไม่ใช่" ในช่องกว้าง 9
    #   - {qtext} : ตัวคำถาม
    print(f"{best_score(qtext):10.2f}  {'ใช่' if in_scope else 'ไม่ใช่':9}  {qtext}")

### แบบฝึกหัดที่ 4 (Exercise 4): การกำหนดเกณฑ์การงดตอบ

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: กำหนดเกณฑ์การงดตอบ (abstention threshold)
# ให้พิจารณาตารางข้างต้น แล้วเลือกค่าที่อยู่ "ระหว่าง" คะแนนของคำถามในขอบเขต (ซึ่งมักสูงกว่า)
# และคะแนนของคำถามนอกขอบเขต (ซึ่งมักต่ำกว่า) คำถามที่มีคะแนนสูงสุดต่ำกว่าเกณฑ์นี้จะถูกงดตอบ
# คำใบ้: เริ่มจาก 0.4 รันเซลล์และพิจารณาผลลัพธ์ จากนั้นปรับค่าให้สูงขึ้นหรือต่ำลง
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้ให้ผู้เรียนกำหนด "เกณฑ์การงดตอบ" (ABSTAIN_BELOW) ซึ่งเป็นกลไกป้องกันด้านความปลอดภัย (guard-rail)
#   แล้วทดสอบกับชุดคำถามฝึกหัด 8 ข้อว่าระบบตัดสินใจ "ตอบ" หรือ "งดตอบ" ได้ถูกต้องกี่ข้อ
#     - ตัดสินถูก = งดตอบคำถามนอกขอบเขต และตอบคำถามในขอบเขต
#     - เกณฑ์สูงเกินไป → งดตอบแม้คำถามที่มีคำตอบ (ไม่สะดวก แต่ปลอดภัย)
#     - เกณฑ์ต่ำเกินไป → คำถามนอกขอบเขตหลุดไปถึง LLM (เสี่ยงได้คำตอบที่ไม่มีหลักฐานรองรับ)
#   ผู้เรียนต้องเติมค่าเกณฑ์แทน ... เพียง 1 จุด
#   ผลลัพธ์ที่จะเห็น: ผลการตัดสินใจของแต่ละคำถาม (ตอบ/งดตอบ, ถูก/ผิด) และจำนวนข้อที่ถูกทั้งหมด
# ============================================================================

# กำหนดเกณฑ์การงดตอบ เก็บไว้ในตัวแปร ABSTAIN_BELOW
#   - ... คือช่องว่างที่ผู้เรียนต้องเติมเป็นตัวเลขทศนิยมระหว่าง 0 ถึง 1 (ดูคำใบ้ด้านบน)
#   - คำถามที่คะแนนความคล้ายคลึงสูงสุด "ต่ำกว่า" ค่านี้ จะถูกงดตอบ
ABSTAIN_BELOW = ...    # TODO: ค่าระหว่าง 0 ถึง 1 เช่น 0.4

# สร้างตัวนับ correct เริ่มต้นที่ 0 เพื่อนับจำนวนคำถามที่ระบบตัดสินใจได้ถูกต้อง
correct = 0
# วนซ้ำทีละคำถามในชุดฝึกหัด (qtext = คำถาม, in_scope = True/False ว่าอยู่ในขอบเขตหรือไม่)
for qtext, in_scope in practice:
    # ตัดสินใจว่าจะงดตอบหรือไม่: เปรียบเทียบคะแนนสูงสุดกับเกณฑ์ด้วยเครื่องหมาย <
    #   ผลลัพธ์เป็น True (งดตอบ) เมื่อคะแนนต่ำกว่าเกณฑ์ หรือ False (ตอบ) เมื่อคะแนนถึงเกณฑ์
    refuse = best_score(qtext) < ABSTAIN_BELOW
    # ตรวจว่าการตัดสินใจถูกต้องหรือไม่ ด้วยเครื่องหมาย != (ไม่เท่ากับ)
    #   - ถูกต้องเมื่อ refuse กับ in_scope มีค่า "ตรงข้ามกัน" คือ
    #     งดตอบ (True) กับคำถามนอกขอบเขต (False) หรือ ตอบ (False) กับคำถามในขอบเขต (True)
    ok = (refuse != in_scope)             # งดตอบเฉพาะคำถามนอกขอบเขต
    # เพิ่มตัวนับ: += หมายถึงบวกค่าเพิ่มเข้าไปในตัวแปรเดิม
    #   ใน Python True มีค่าเท่ากับ 1 และ False เท่ากับ 0 จึงบวก 1 เฉพาะเมื่อตัดสินถูก
    correct += ok
    # พิมพ์ผลของคำถามนี้ 1 บรรทัด
    #   - {'งดตอบ' if refuse else 'ตอบ':6} : แสดง "งดตอบ" หรือ "ตอบ" ในช่องกว้าง 6 ตัวอักษร
    #   - {'ถูก ' if ok else 'ผิด '}        : แสดง "ถูก" หรือ "ผิด" ตามผลการตรวจ
    #   - {qtext}                          : ตัวคำถาม
    print(f"{'งดตอบ' if refuse else 'ตอบ':6}  {'ถูก ' if ok else 'ผิด '}  {qtext}")
# หลังวนครบทุกข้อ (บรรทัดนี้ไม่ย่อหน้า จึงอยู่นอก for และทำงานครั้งเดียว) พิมพ์สรุปจำนวนข้อที่ถูก
#   - {correct}       : จำนวนข้อที่ตัดสินถูก
#   - len(practice)   : ฟังก์ชัน len นับจำนวนสมาชิกในลิสต์ = จำนวนคำถามทั้งหมด (8)
print(f"\nความแม่นยำของการงดตอบในชุดคำถามฝึกหัด: {correct}/{len(practice)}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# เลือกค่าโดยอัตโนมัติ: ทดสอบเกณฑ์ที่อยู่กึ่งกลางระหว่างคะแนนที่อยู่ติดกัน แล้วเลือกค่าที่ให้จำนวนข้อถูกมากที่สุด
# (หากเท่ากัน ให้เลือกเกณฑ์ที่ต่ำกว่า ซึ่งงดตอบน้อยกว่า)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 4)
#   แทนที่จะเดาเกณฑ์การงดตอบ เซลล์นี้ "ค้นหาเกณฑ์ที่ดีที่สุดโดยอัตโนมัติ" จากชุดคำถามฝึกหัด
#   1) คำนวณคะแนนสูงสุดของทุกคำถาม แล้วเรียงจากน้อยไปมาก
#   2) สร้างรายการเกณฑ์ที่เป็นไปได้ = 0.0 และค่ากึ่งกลางระหว่างคะแนนที่อยู่ติดกัน
#   3) นับว่าแต่ละเกณฑ์ตัดสินถูกกี่ข้อ แล้วเลือกเกณฑ์ที่ถูกมากที่สุด (ถ้าเท่ากัน เลือกค่าที่ต่ำกว่า)
#   4) ทดสอบเกณฑ์ที่ได้กับชุดคำถามฝึกหัด เหมือนในแบบฝึกหัด
#   ผลลัพธ์ที่จะเห็น: ค่า ABSTAIN_BELOW ที่เลือกได้, ผลตอบ/งดตอบของแต่ละข้อ และจำนวนข้อที่ถูก
#   (ข้อควรระวัง: การเลือกเกณฑ์จากข้อมูลเพียง 8 ข้อ อาจใช้ไม่ได้ดีกับคำถามชุดอื่น)
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: หาเกณฑ์ที่ดีที่สุดโดยอัตโนมัติ
# ----------------------------------------------------------------------------
# สร้างลิสต์ scored ของคู่ (คะแนนสูงสุด, อยู่ในขอบเขตหรือไม่) แล้วเรียงลำดับ
#   - (best_score(q), s) for q, s in practice : วนทุกคำถาม q พร้อมค่า s (True/False)
#                                               แล้วสร้างคู่ (คะแนน, s)
#   - sorted(...) : ฟังก์ชันของ Python เรียงลำดับจากน้อยไปมาก (คู่จะเรียงตามค่าแรก คือคะแนน)
scored = sorted((best_score(q), s) for q, s in practice)
# สร้างลิสต์ candidates ของ "เกณฑ์ที่เป็นไปได้"
#   - [0.0]                       : เกณฑ์ 0.0 (แทบไม่งดตอบเลย) เป็นตัวเลือกแรก
#   - zip(scored, scored[1:])     : จับคู่สมาชิกแต่ละตัวกับตัวถัดไป
#                                   (scored[1:] คือลิสต์ตั้งแต่ตัวที่ 1 ไปจนจบ = เลื่อนไป 1 ตำแหน่ง)
#   - a, b                        : คู่คะแนนที่อยู่ติดกัน (a[0], b[0] คือคะแนนของแต่ละตัว)
#   - (a[0] + b[0]) / 2           : ค่ากึ่งกลางระหว่างคะแนนที่อยู่ติดกัน
#   - round(..., 3)               : ปัดเศษเป็นทศนิยม 3 ตำแหน่ง
#   - เครื่องหมาย + ระหว่างลิสต์    : นำ 2 ลิสต์มาต่อกันเป็นลิสต์เดียว
candidates = [0.0] + [round((a[0] + b[0]) / 2, 3) for a, b in zip(scored, scored[1:])]
# ฟังก์ชัน n_right(t) — นับว่าถ้าใช้เกณฑ์ t จะตัดสินถูกกี่ข้อ
#   - t : ค่าเกณฑ์ที่ต้องการทดสอบ
def n_right(t):
    # สำหรับทุกคู่ (sc = คะแนน, s = อยู่ในขอบเขตหรือไม่) ใน scored
    #   - (sc < t)       : True ถ้าจะงดตอบ (คะแนนต่ำกว่าเกณฑ์)
    #   - ... != s       : ถูกต้องเมื่อการงดตอบตรงข้ามกับการอยู่ในขอบเขต (งดตอบเฉพาะคำถามนอกขอบเขต)
    #   - sum(...)       : รวมผล โดย True นับเป็น 1 และ False นับเป็น 0 = จำนวนข้อที่ถูก
    return sum((sc < t) != s for sc, s in scored)
# เลือกเกณฑ์ที่ดีที่สุดจาก candidates เก็บไว้ใน ABSTAIN_BELOW
#   - max(candidates, key=...) : ฟังก์ชันของ Python หาสมาชิกที่ "มากที่สุด" ตามเกณฑ์ที่ระบุใน key
#   - lambda t: (n_right(t), -t) : lambda คือการสร้างฟังก์ชันสั้น ๆ ในบรรทัดเดียว
#       รับ t แล้วคืนคู่ (จำนวนข้อที่ถูก, ค่าติดลบของ t)
#       Python เปรียบเทียบคู่ทีละค่า: ค่าแรกมากกว่าชนะ (ถูกมากที่สุด)
#       ถ้าเท่ากัน ดูค่าที่สอง -t ซึ่งมากกว่าเมื่อ t น้อยกว่า จึงได้เกณฑ์ที่ต่ำกว่า (งดตอบน้อยกว่า)
ABSTAIN_BELOW = max(candidates, key=lambda t: (n_right(t), -t))
# แสดงค่าเกณฑ์ที่เลือกได้
print("ABSTAIN_BELOW =", ABSTAIN_BELOW)

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ทดสอบเกณฑ์ที่เลือกได้กับชุดคำถามฝึกหัด (เหมือนในแบบฝึกหัด)
# ----------------------------------------------------------------------------
# ตัวนับจำนวนข้อที่ตัดสินถูก เริ่มที่ 0
correct = 0
# วนทีละคำถาม (qtext = คำถาม, in_scope = อยู่ในขอบเขตหรือไม่)
for qtext, in_scope in practice:
    # งดตอบ (True) เมื่อคะแนนสูงสุดต่ำกว่าเกณฑ์
    refuse = best_score(qtext) < ABSTAIN_BELOW
    # ตัดสินถูกเมื่อ refuse กับ in_scope มีค่าตรงข้ามกัน (!= คือ "ไม่เท่ากับ")
    ok = (refuse != in_scope)
    # เพิ่มตัวนับ 1 เมื่อถูก (True = 1, False = 0)
    correct += ok
    # พิมพ์ผล: "งดตอบ"/"ตอบ" (ช่องกว้าง 6), "ถูก"/"ผิด" และคำถาม
    print(f"{'งดตอบ' if refuse else 'ตอบ':6}  {'ถูก ' if ok else 'ผิด '}  {qtext}")
# สรุปจำนวนข้อที่ถูกจากทั้งหมด (len(practice) = จำนวนคำถาม)
print(f"\nความแม่นยำของการงดตอบในชุดคำถามฝึกหัด: {correct}/{len(practice)}")

💡 **คำอธิบายผลลัพธ์** ขั้นตอนนี้เป็นการกำหนดกลไกป้องกันด้านความปลอดภัย (guard-rail) เกณฑ์ที่**สูงเกินไป**จะทำให้ระบบงดตอบคำถามที่แนวทางเวชปฏิบัติ*มี*คำตอบ (ไม่สะดวกต่อผู้ใช้ แต่ปลอดภัย) ส่วนเกณฑ์ที่**ต่ำเกินไป**จะทำให้คำถามที่ไม่เกี่ยวข้องผ่านไปถึง LLM ซึ่งอาจสร้างคำตอบที่ไม่มีหลักฐานรองรับ (มีความเสี่ยง) เซลล์ถัดไปรวมกลไกป้องกันนี้เข้ากับ LLM

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้รวม "กลไกป้องกัน" (guard-rail) จากแบบฝึกหัดที่ 4 เข้ากับระบบ RAG
#   ผ่านฟังก์ชัน answer_safely(question) ซึ่งทำงาน 2 ชั้น:
#     ชั้นที่ 1 (ไม่ใช้ LLM) : ถ้าคะแนนความคล้ายคลึงสูงสุดต่ำกว่าเกณฑ์ ABSTAIN_BELOW → งดตอบทันที
#     ชั้นที่ 2 (ใช้ LLM)   : ถ้าผ่านเกณฑ์ → ตอบแบบ RAG ซึ่ง LLM ยังสามารถงดตอบเองได้อีกชั้นหนึ่ง
#   ผลลัพธ์ที่จะเห็น: คำตอบของคำถามในขอบเขต (เรื่อง statin) และการงดตอบคำถามนอกขอบเขต (เรื่องหิด)
# ============================================================================

# ----------------------------------------------------------------------------
# ฟังก์ชัน answer_safely — ตอบอย่างปลอดภัย (คัดกรองด้วยเกณฑ์ก่อนส่งให้ LLM)
#   - question : คำถามของผู้ใช้
#   - k=3      : จำนวนข้อความที่ค้นคืนมาเป็นบริบท (ค่าเริ่มต้น 3)
# ----------------------------------------------------------------------------
def answer_safely(question, k=3):
    # ตรวจสอบเกณฑ์ก่อน (ไม่ใช้ LLM) แล้วจึงตอบแบบ RAG ที่อ้างอิงเอกสาร (LLM ยังสามารถงดตอบเองได้)
    # if: ถ้าคะแนนความคล้ายคลึงสูงสุดของคำถามต่ำกว่าเกณฑ์ (คำถามน่าจะอยู่นอกขอบเขต)
    if best_score(question) < ABSTAIN_BELOW:
        # คืนวลีงดตอบ ต่อท้าย (+) ด้วยเหตุผลภาษาไทยในวงเล็บ แล้วจบฟังก์ชันทันที (ไม่เรียก LLM)
        return ABSTAIN_PHRASE + " (คะแนนความคล้ายคลึงสูงสุดต่ำกว่าเกณฑ์)"
    # ถ้าผ่านเกณฑ์ (ไม่เข้า if ข้างบน) ให้ตอบด้วยระบบ RAG ตามปกติ
    #   - question : คำถาม
    #   - k        : จำนวนข้อความที่ใช้เป็นบริบท
    return answer_with_rag(question, k)

# ทดสอบคำถามในขอบเขต 1 ข้อและนอกขอบเขต 1 ข้อ (ผู้เรียนสามารถเปลี่ยนเป็นคำถามของตนเองได้)
# วนซ้ำทีละคำถามในลิสต์ (คำสั่งยาว 2 บรรทัดเพราะลิสต์ในวงเล็บ [ ] เขียนต่อบรรทัดได้)
#   - "Should a 60-year-old with diabetes take a statin?" : ในขอบเขต (ผู้ป่วยเบาหวานอายุ 60 ควรใช้ statin หรือไม่)
#   - "How is scabies treated?"                           : นอกขอบเขต (การรักษาโรคหิด)
#   - qtext : ตัวแปรที่รับคำถามทีละข้อในแต่ละรอบ
for qtext in ["Should a 60-year-old with diabetes take a statin?",
              "How is scabies treated?"]:
    # พิมพ์คำถาม และคำตอบจาก answer_safely(qtext)
    #   - "\n" คือขึ้นบรรทัดใหม่ เพื่อแยกคำถาม คำตอบ และเว้นบรรทัดระหว่างแต่ละข้อ
    print("คำถาม:", qtext, "\nคำตอบ:", answer_safely(qtext), "\n")

✍️ **คำถามที่ 3** โปรดระบุเกณฑ์ที่เลือกและจำนวนคำถามฝึกหัดที่ระบบตัดสินได้ถูกต้อง จากนั้นจงอธิบายใน 2–3 ประโยคว่า ในมุมมองความปลอดภัยของผู้ป่วย ความผิดพลาดประเภทใดร้ายแรงกว่าสำหรับระบบผู้ช่วยด้านแนวทางเวชปฏิบัติ ระหว่างการงดตอบคำถามที่ตอบได้ กับการตอบคำถามที่อยู่นอกแนวทางเวชปฏิบัติ เพราะเหตุใด

✍️ **คำตอบ:**

## จาก RAG สู่เอเจนต์ปัญญาประดิษฐ์ (AI Agent)

**เอเจนต์ปัญญาประดิษฐ์ (AI agent)** สามารถ*ตัดสินใจ*เลือกใช้ **เครื่องมือ (tool)** เช่น การค้นแนวทางเวชปฏิบัติ เครื่องคำนวณ การตรวจสอบอันตรกิริยาระหว่างยา (drug interaction) หรือการดึงข้อมูลจากระบบเวชระเบียนอิเล็กทรอนิกส์ (EHR) แล้วดำเนินการตามผลลัพธ์ ซึ่งมักทำซ้ำหลายรอบ เซลล์ด้านล่างเป็นตัวอย่างการเลือกเครื่องมืออย่างง่ายโดยไม่ใช้ framework เพื่อแสดงหลักการ ประกอบด้วยเครื่องมือคำนวณ BMI และ RAG แบบมีกลไกป้องกันเป็นทางเลือกสำรอง

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้สาธิตหลักการของ "เอเจนต์ปัญญาประดิษฐ์" (AI agent) อย่างง่าย ซึ่งสามารถ "เลือกเครื่องมือ" (tool)
#   ให้เหมาะกับคำขอได้ โดยมีเครื่องมือ 2 ชนิด:
#     1) เครื่องคำนวณ BMI (ดัชนีมวลกาย) — ให้ผลการคำนวณแม่นยำ ขณะที่ LLM มักคำนวณผิด
#     2) RAG แนวทางเวชปฏิบัติแบบมีกลไกป้องกัน (answer_safely) — เป็นทางเลือกสำรองสำหรับคำถามทางคลินิก
#   เอเจนต์ในเซลล์นี้เลือกเครื่องมือด้วย "คำสำคัญ" (keyword) อย่างง่าย ส่วนเอเจนต์จริงจะให้ LLM เป็นผู้เลือก
#   ผลลัพธ์ที่จะเห็น: คำตอบ 2 ข้อ พร้อมป้ายบอกว่าใช้เครื่องมือใด
# ============================================================================

# นำเข้าไลบรารี re (regular expression) ซึ่งมีมากับ Python
#   ใช้ค้นหา "รูปแบบ" ในข้อความ เช่น ดึงตัวเลขทั้งหมดออกจากประโยค
import re

# ----------------------------------------------------------------------------
# เครื่องมือที่ 1: bmi_tool — คำนวณดัชนีมวลกาย (Body Mass Index: BMI)
#   - weight_kg : น้ำหนัก (กิโลกรัม)
#   - height_m  : ส่วนสูง (เมตร)
# ----------------------------------------------------------------------------
def bmi_tool(weight_kg, height_m):
    # คำนวณ BMI = น้ำหนัก ÷ (ส่วนสูง ยกกำลัง 2) แล้วคืนค่า
    #   - ** คือการยกกำลังใน Python (height_m ** 2 = ส่วนสูงยกกำลังสอง)
    #   - / คือการหาร
    #   - round(..., 1) : ปัดเศษเป็นทศนิยม 1 ตำแหน่ง
    return round(weight_kg / (height_m ** 2), 1)

# ----------------------------------------------------------------------------
# เอเจนต์อย่างง่าย: simple_agent — เลือกเครื่องมือตามเนื้อหาของคำถาม
#   - question : คำขอ/คำถามของผู้ใช้
# ----------------------------------------------------------------------------
def simple_agent(question):
    # เลือกเครื่องมือด้วยคำสำคัญอย่างง่าย ส่วนเอเจนต์จริงจะให้ LLM สร้างคำสั่งเรียกเครื่องมือแบบมีโครงสร้าง (เช่น JSON)
    # if: ตรวจว่าคำถามมีคำว่า "bmi" อยู่หรือไม่
    #   - question.lower() : เมธอดของข้อความ แปลงเป็นตัวพิมพ์เล็กทั้งหมด (จึงตรวจพบทั้ง "BMI" และ "bmi")
    #   - "bmi" in ...     : คืน True ถ้าพบคำว่า "bmi" อยู่ในข้อความ
    if "bmi" in question.lower():
        # ดึงตัวเลขทั้งหมดในคำถามออกมาเป็นลิสต์ของเลขทศนิยม เก็บไว้ใน nums
        #   - re.findall(pattern, question) : ค้นหาทุกส่วนของข้อความที่ตรงรูปแบบ คืนเป็นลิสต์ของข้อความ
        #   - r"\d+(?:\.\d+)?" : รูปแบบ (regular expression) ของตัวเลข
        #       * r"..."     : raw string ให้ Python อ่านเครื่องหมาย \ ตามตัวอักษร
        #       * \d+        : ตัวเลข 0-9 ติดกันอย่างน้อย 1 ตัว เช่น 82
        #       * (?:\.\d+)? : ส่วนทศนิยม (จุด ตามด้วยตัวเลข) ซึ่งมีหรือไม่มีก็ได้ เช่น .75
        #   - float(x) for x in ... : แปลงข้อความตัวเลขแต่ละตัวเป็นเลขทศนิยม
        #   ตัวอย่าง: "weight 82 kg and height 1.75 m" → [82.0, 1.75]
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", question)]
        # if: ตรวจว่าพบตัวเลขอย่างน้อย 2 ตัว (น้ำหนักและส่วนสูง) — len(nums) คือจำนวนสมาชิกในลิสต์
        if len(nums) >= 2:
            # เรียกเครื่องมือคำนวณ BMI แล้วคืนผลพร้อมป้ายบอกชื่อเครื่องมือ
            #   - nums[0] : ตัวเลขตัวแรก ถือเป็นน้ำหนัก (กก.)
            #   - nums[1] : ตัวเลขตัวที่สอง ถือเป็นส่วนสูง (ม.)
            #   (ข้อจำกัด: วิธีนี้สมมติว่าผู้ใช้ระบุน้ำหนักก่อนส่วนสูงเสมอ)
            return f"[เครื่องมือ: คำนวณ BMI] BMI = {bmi_tool(nums[0], nums[1])} kg/m2"
    # กรณีไม่ใช่คำถาม BMI (หรือพบตัวเลขไม่ครบ 2 ตัว) ให้ใช้เครื่องมือสำรอง:
    #   RAG แนวทางเวชปฏิบัติแบบมีกลไกป้องกัน answer_safely(question) แล้วต่อท้ายป้ายชื่อเครื่องมือด้วย +
    return "[เครื่องมือ: RAG แนวทางเวชปฏิบัติ] " + answer_safely(question)

# ทดสอบเอเจนต์กับคำขอคำนวณ BMI (น้ำหนัก 82 กก. ส่วนสูง 1.75 ม.) → ควรเลือกเครื่องคำนวณ BMI
print(simple_agent("Compute BMI for weight 82 kg and height 1.75 m."))
# ทดสอบเอเจนต์กับคำถามทางคลินิกเรื่อง statin → ควรเลือก RAG แนวทางเวชปฏิบัติ
print(simple_agent("Should diabetic patients aged 40-75 take statins?"))

💡 **คำอธิบายผลลัพธ์** เอเจนต์พิจารณาคำขอ แล้วเลือกเครื่องคำนวณสำหรับคำถามเรื่อง BMI (ให้ผลการคำนวณที่แม่นยำ ขณะที่ LLM มักคำนวณผิดพลาด) และเลือก RAG แนวทางเวชปฏิบัติสำหรับคำถามทางคลินิก framework สำหรับเอเจนต์ที่ใช้งานจริงทำงานในลักษณะเดียวกัน แต่ให้ LLM เป็นผู้เลือกเครื่องมือและข้อมูลนำเข้า ซึ่งยืดหยุ่นกว่า ทั้งนี้จึงจำเป็นต้องบันทึกและตรวจสอบการดำเนินการของเอเจนต์ทุกครั้ง

✍️ **คำถามที่ 4** โปรดระบุเครื่องมือ 1 ประเภทที่ผู้เรียนเห็นว่าควรให้เอเจนต์ปัญญาประดิษฐ์ทางคลินิกในหน่วยงานของตนใช้ได้ (เช่น การตรวจสอบอันตรกิริยาระหว่างยา การเรียกดูผลการตรวจทางห้องปฏิบัติการ การนัดหมาย) และความเสี่ยงด้านความปลอดภัย 1 ประการ หากให้เอเจนต์ใช้เครื่องมือนั้นโดยอัตโนมัติ

✍️ **คำตอบ:**

✍️ **คำถามที่ 5 (สรุป)** จงอธิบายใน 2–3 ประโยคว่า การให้ระบบปัญญาประดิษฐ์ตอบโดยอ้างอิงแนวทางเวชปฏิบัติที่ค้นคืนได้ (พร้อมการอ้างอิงแหล่งที่มาและการงดตอบ) ส่งผลต่อความน่าเชื่อถือของเครื่องมือดังกล่าวในเวชปฏิบัติอย่างไร และประเด็นใดที่ผู้ใช้ยังต้องตรวจสอบด้วยตนเอง

✍️ **คำตอบ:**

## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** ผู้เรียนสามารถทำปฏิบัติการให้เสร็จสมบูรณ์โดยไม่ต้องทำส่วนนี้ โค้ดตั้งต้น (baseline) ด้านล่างสามารถรันได้ทันทีและสร้างไฟล์ผลลัพธ์ในรูปแบบที่ถูกต้อง ผู้เรียนทุกคนจึง*สามารถ*ส่งผลได้ ส่วนผู้เรียนที่เขียนโปรแกรมได้สามารถพัฒนาระบบให้ได้คะแนนสูงกว่า baseline

รายวิชาจัดเตรียม **คลังความรู้แนวทางเวชปฏิบัติสังเคราะห์ 40 ข้อความ** (`rag/kb.json`, รหัส `P01`–`P40`) และ **คำถาม 22 ข้อ** (`rag/questions.json`) ซึ่ง**บางข้อ*ไม่สามารถ*ตอบได้จากคลังความรู้นี้** สำหรับคำถามแต่ละข้อ ให้ส่งรหัสข้อความ 3 อันดับแรกที่ค้นคืนได้และคำตอบที่ระบบสร้างขึ้น ในไฟล์ `lab09_submission.json` ตามรูปแบบต่อไปนี้

```json
{"Q01": {"passages": ["P12", "P03", "P30"], "answer": "..."}, ...}
```

**ตัวชี้วัด: 0.5 × Recall@3** (สำหรับคำถามในขอบเขต: มีข้อความที่ถูกต้องอยู่ใน 3 อันดับแรกหรือไม่) **+ 0.5 × ความแม่นยำของการงดตอบ (abstention accuracy)** (คำตอบมีวลี *"Not in the guidelines"* เฉพาะเมื่อคำถามอยู่นอกขอบเขตหรือไม่)

หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่า ด้านบน แล้วรันเซลล์นั้นอีกครั้ง เซลล์ baseline จะ**แทนที่** `kb` และ `index` ขนาดเล็กที่ใช้ก่อนหน้า ดังนั้น `search()`, `answer_with_rag()` และ `answer_safely()` จะทำงานบนคลังความรู้ใหม่โดยอัตโนมัติ

**หลักความซื่อสัตย์ทางวิชาการ (honor code):** ผลที่ส่งต้องได้จากระบบ (pipeline) ของผู้เรียนเอง ห้ามกำหนดป้ายกำกับของคำถามทดสอบทั้ง 22 ข้อด้วยตนเอง และห้ามเลือกรหัสข้อความหรือคำตอบให้คำถามเหล่านั้นด้วยตนเอง

In [ ]:
# 🏆 Baseline — ค้นคืนข้อความและตอบคำถามของรายวิชา 22 ข้อ (รันได้ทันที ใช้เวลาประมาณ 2-4 นาทีบน T4 GPU)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (โจทย์ท้าทาย — ไม่บังคับ)
#   เซลล์นี้เป็นโค้ดตั้งต้น (baseline) สำหรับตารางอันดับ (leaderboard) ทำงานดังนี้
#   1) ดาวน์โหลดคลังความรู้ใหม่ 40 ข้อความ (รหัส P01–P40) และคำถาม 22 ข้อของรายวิชา
#   2) แทนที่คลังความรู้ kb และดัชนี index ขนาดเล็กเดิม ด้วยคลังความรู้ใหม่
#      (ฟังก์ชัน search, answer_with_rag, answer_safely จึงทำงานบนคลังใหม่โดยอัตโนมัติ)
#   3) สำหรับคำถามแต่ละข้อ: ค้นคืนรหัสข้อความ 3 อันดับแรก แล้วตัดสินใจงดตอบหรือตอบ
#   4) บันทึกผลเป็นไฟล์ lab09_submission.json และส่งไปยังตารางอันดับ (ถ้ากรอก NAME และ JOIN_CODE แล้ว)
#   ตัวชี้วัด: 0.5 × Recall@3 (ข้อความที่ถูกต้องอยู่ใน 3 อันดับแรกหรือไม่) + 0.5 × ความแม่นยำของการงดตอบ
#   ผลลัพธ์ที่จะเห็น: รหัสคำถาม คะแนนสูงสุด และคำตอบย่อ ทีละข้อ แล้วข้อความยืนยันการบันทึก/ส่งไฟล์
# ============================================================================

# ----------------------------------------------------------------------------
# ส่วนที่ 1: โหลดคลังความรู้และคำถามของรายวิชา แล้วสร้างดัชนีใหม่
# ----------------------------------------------------------------------------
# นำเข้าไลบรารี json (มากับ Python) สำหรับอ่าน/เขียนไฟล์ JSON
#   JSON เป็นรูปแบบไฟล์ข้อความที่เก็บข้อมูลแบบลิสต์และดิกชันนารี
import json
# โหลดคลังความรู้ของรายวิชา เก็บไว้ใน kb_docs (ทำจากในวงเล็บออกมาด้านนอก)
#   1) lab_data("rag/kb.json") : ฟังก์ชันจากเซลล์การตั้งค่า ดาวน์โหลดไฟล์และคืนตำแหน่งไฟล์ในเครื่อง
#   2) open(..., encoding="utf-8") : เปิดไฟล์เพื่ออ่าน โดยใช้รหัสอักขระ UTF-8 (รองรับทุกภาษา)
#   3) json.load(...) : อ่านเนื้อหา JSON แปลงเป็นข้อมูล Python
#   ผลลัพธ์เป็นลิสต์ของดิกชันนารี แต่ละตัวมีกุญแจ "id" (รหัสข้อความ) และ "text" (ตัวข้อความ)
kb_docs   = json.load(open(lab_data("rag/kb.json"), encoding="utf-8"))
# โหลดคำถาม 22 ข้อในลักษณะเดียวกัน เก็บไว้ใน questions
#   แต่ละข้อเป็นดิกชันนารีที่มีกุญแจ "id" (รหัสคำถาม เช่น Q01) และ "question" (ตัวคำถาม)
questions = json.load(open(lab_data("rag/questions.json"), encoding="utf-8"))
# สร้างลิสต์ kb_ids เก็บเฉพาะรหัสข้อความ (เช่น "P01") ตามลำดับเดียวกับคลังความรู้
#   - for d in kb_docs : วนทีละเอกสาร d
#   - d["id"]          : หยิบค่าของกุญแจ "id" จากดิกชันนารี
kb_ids = [d["id"] for d in kb_docs]
# สร้างลิสต์ kb ใหม่ เก็บเฉพาะตัวข้อความ (d["text"]) — เขียนทับตัวแปร kb เดิม
kb = [d["text"] for d in kb_docs]                        # แทนที่คลังความรู้ขนาดเล็กก่อนหน้า
# แปลงข้อความทุกข้อในคลังใหม่เป็นเวกเตอร์
#   - kb                        : ลิสต์ข้อความ
#   - normalize_embeddings=True : ปรับความยาวเวกเตอร์เป็น 1 เพื่อให้ผลคูณภายในเท่ากับ cosine similarity
emb = embedder.encode(kb, normalize_embeddings=True)
# บรรทัดนี้มี 2 คำสั่งคั่นด้วยเครื่องหมาย ; (เซมิโคลอน = เขียนหลายคำสั่งในบรรทัดเดียว)
#   1) faiss.IndexFlatIP(emb.shape[1]) : สร้างดัชนีใหม่ (วัดด้วยผลคูณภายใน) ขนาดเท่าจำนวนมิติของเวกเตอร์
#   2) index.add(emb)                  : ใส่เวกเตอร์ของคลังความรู้ใหม่ลงในดัชนี
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)

# ----------------------------------------------------------------------------
# ส่วนที่ 2: ค่าที่ผู้เรียนปรับได้
# ----------------------------------------------------------------------------
# เกณฑ์การงดตอบสำหรับคลังความรู้นี้: คำถามที่คะแนนสูงสุดต่ำกว่า 0.35 จะถูกงดตอบ
LB_ABSTAIN_BELOW = 0.35   # เกณฑ์ความคล้ายคลึงสำหรับคลังความรู้นี้ (ค่าจากส่วนหลักอาจใช้กับคลังนี้ไม่ได้) ปรับค่าได้ตามต้องการ
# เลือกว่าจะใช้ LLM สร้างคำตอบหรือไม่: True = ใช้ (ช้ากว่า), False = ไม่ใช้
USE_LLM = True            # False = ไม่ใช้ LLM (เร็วกว่ามาก เนื่องจากคะแนนพิจารณาเฉพาะการตัดสินใจงดตอบ)

# ----------------------------------------------------------------------------
# ส่วนที่ 3: ตอบคำถามทีละข้อ และเก็บผลลัพธ์
# ----------------------------------------------------------------------------
# สร้างดิกชันนารีว่าง { } ชื่อ submission สำหรับเก็บผลของทุกคำถาม (กุญแจ = รหัสคำถาม)
submission = {}
# วนซ้ำทีละคำถาม q (แต่ละ q เป็นดิกชันนารีที่มี "id" และ "question")
for q in questions:
    # แปลงคำถามเป็นเวกเตอร์ (ใส่ใน [ ] เพื่อให้ได้ตาราง 1 แถว) และ normalize เช่นเดียวกับคลังความรู้
    #   - q["question"] : หยิบตัวคำถามจากดิกชันนารี
    qv = embedder.encode([q["question"]], normalize_embeddings=True)
    # ค้นหาข้อความ 3 อันดับแรกในดัชนี
    #   - qv : เวกเตอร์คำถาม
    #   - 3  : จำนวนข้อความที่ต้องการ (top-3 ตามตัวชี้วัด Recall@3)
    #   ได้ scores (คะแนน) และ idx (ลำดับที่ของข้อความ) เป็นตาราง 1 แถว
    scores, idx = index.search(qv, 3)
    # แปลงลำดับที่ของข้อความเป็นรหัสข้อความ (เช่น "P12") เก็บไว้ในลิสต์ top
    #   - idx[0]    : แถวแรก (แถวเดียว) ของผลลัพธ์
    #   - kb_ids[i] : รหัสของข้อความลำดับที่ i
    top = [kb_ids[i] for i in idx[0]]
    # if: ถ้าคะแนนของข้อความอันดับ 1 (scores[0][0] = แถวแรก ค่าแรก) ต่ำกว่าเกณฑ์ → งดตอบ
    if scores[0][0] < LB_ABSTAIN_BELOW:
        # ใช้วลีงดตอบมาตรฐาน "Not in the guidelines." เป็นคำตอบ
        answer = ABSTAIN_PHRASE
    # elif (ย่อมาจาก else if): ถ้าไม่เข้าเงื่อนไขแรก และ USE_LLM เป็น True → ให้ LLM ตอบแบบ RAG
    elif USE_LLM:
        # ตอบด้วย answer_with_rag (ใช้ดัชนีใหม่ที่สร้างในเซลล์นี้)
        answer = answer_with_rag(q["question"])           # LLM อาจตัดสินใจตอบ 'Not in the guidelines' เอง
    # else: กรณีอื่น ๆ ที่เหลือ (ผ่านเกณฑ์ แต่ไม่ใช้ LLM)
    else:
        # สร้างคำตอบอย่างง่ายโดยระบุรหัสข้อความที่ค้นคืนได้
        #   - ", ".join(top) : ต่อรหัสในลิสต์ top เป็นข้อความเดียว คั่นด้วย ", " เช่น "P12, P03, P30"
        answer = "See passages " + ", ".join(top)
    # บันทึกผลของคำถามนี้ลงใน submission โดยใช้รหัสคำถามเป็นกุญแจ
    #   - submission[q["id"]] = ... : เพิ่ม/กำหนดค่าในดิกชันนารีด้วยกุญแจ q["id"] (เช่น "Q01")
    #   - ค่าเป็นดิกชันนารีย่อย 2 กุญแจ: "passages" (รหัสข้อความ 3 อันดับ) และ "answer" (คำตอบ)
    submission[q["id"]] = {"passages": top, "answer": answer}
    # พิมพ์ความคืบหน้า: รหัสคำถาม, คะแนนสูงสุด และคำตอบย่อ
    #   - round(float(scores[0][0]), 2) : แปลงคะแนนอันดับ 1 เป็นเลขทศนิยม แล้วปัดเหลือ 2 ตำแหน่ง
    #   - answer[:80]                   : ตัดเอาเฉพาะ 80 ตัวอักษรแรกของคำตอบ ([:80] คือการตัดช่วงข้อความ)
    #   - .replace("\n", " ")           : แทนที่การขึ้นบรรทัดใหม่ด้วยช่องว่าง ให้แสดงเป็นบรรทัดเดียว
    print(q["id"], round(float(scores[0][0]), 2), answer[:80].replace("\n", " "))

# ----------------------------------------------------------------------------
# ส่วนที่ 4: บันทึกไฟล์ผลลัพธ์ และส่งไปยังตารางอันดับ
# ----------------------------------------------------------------------------
# เขียน submission ลงไฟล์ JSON
#   - json.dump(ข้อมูล, ไฟล์, ...) : แปลงข้อมูล Python เป็น JSON แล้วเขียนลงไฟล์
#   - open("lab09_submission.json", "w", encoding="utf-8") : เปิด (หรือสร้าง) ไฟล์เพื่อเขียน
#       * "w"              : โหมดเขียน (write) — ถ้ามีไฟล์เดิมจะเขียนทับ
#       * encoding="utf-8" : รหัสอักขระ UTF-8
#   - indent=1 : จัดย่อหน้า 1 ช่องในไฟล์ เพื่อให้อ่านง่าย
json.dump(submission, open("lab09_submission.json", "w", encoding="utf-8"), indent=1)
# แสดงข้อความยืนยันการบันทึก พร้อมจำนวนคำตอบ (len(submission) = จำนวนกุญแจในดิกชันนารี)
print("\nบันทึกไฟล์ lab09_submission.json แล้ว จำนวนคำตอบ", len(submission), "ข้อ")
# if: ถ้ากรอกทั้ง NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้ว (ข้อความไม่ว่าง ถือเป็นจริง)
if NAME and JOIN_CODE:
    # ส่งไฟล์ผลลัพธ์ไปยังตารางอันดับ ด้วยฟังก์ชันจากเซลล์การตั้งค่า (จะแสดงคะแนนและอันดับ)
    submit_to_leaderboard("lab09_submission.json")
# else: ถ้ายังไม่ได้กรอก NAME หรือ JOIN_CODE
else:
    # แจ้งเตือนให้กรอก NAME และ JOIN_CODE แล้วรันเซลล์การตั้งค่าอีกครั้งก่อนส่งผล
    print("โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า (และรันเซลล์นั้นอีกครั้ง) เพื่อส่งผลไปยังตารางอันดับ")

**แนวทางการพัฒนาให้ได้คะแนนสูงกว่า baseline**

- **ปรับเกณฑ์การงดตอบ** — แสดงคะแนนสูงสุดของคำถามแต่ละข้อ แล้วพิจารณาการกระจายของคะแนน (โดยไม่กำหนดป้ายกำกับของคำถามทดสอบด้วยตนเอง)
- **ใช้ embedding ที่มีประสิทธิภาพสูงกว่า** — เช่น `BAAI/bge-small-en-v1.5` หรือ `intfloat/e5-small-v2` (โปรดศึกษา model card เนื่องจากบางแบบจำลองกำหนดให้เติมคำนำหน้าคำถาม)
- **การค้นคืนแบบผสม (hybrid retrieval)** — รวมคะแนนคำสำคัญแบบ BM25 (`pip install rank_bm25`) เข้ากับคะแนนความคล้ายคลึงแบบ dense
- **การจัดอันดับใหม่ (re-ranking)** — ให้คะแนนข้อความ 10 อันดับแรกใหม่ด้วย cross-encoder เช่น `cross-encoder/ms-marco-MiniLM-L-6-v2`
- **ให้ LLM ประเมินความเกี่ยวข้อง** — ใช้ prompt ที่เข้มงวดขึ้น หรือถามแบบจำลองก่อนว่า "Is the answer contained in the context? yes/no" แล้วจึงให้ตอบ
- คำตอบของคำถามนอกขอบเขตต้องมีวลี *"Not in the guidelines"* ตรงตามที่กำหนด

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ไม่บังคับ)
#   เซลล์นี้สำหรับผู้เรียนที่พัฒนาระบบ RAG ของตนเองให้ดีกว่า baseline แล้วต้องการส่งผลใหม่
#   ขณะนี้คำสั่งส่งผลถูกปิดไว้ด้วยเครื่องหมาย # (เป็นคอมเมนต์) จึงยังไม่มีการทำงานใด ๆ เมื่อรัน
#   วิธีใช้: สร้างไฟล์ lab09_submission.json ใหม่ในรูปแบบเดียวกับ baseline
#   จากนั้นลบเครื่องหมาย # หน้าบรรทัด submit_to_leaderboard(...) แล้วรันเซลล์
#   (ต้องกรอก NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน)
# ============================================================================

# (ไม่บังคับ) ระบบที่ผู้เรียนพัฒนาเอง: สร้างไฟล์ lab09_submission.json ใหม่ในรูปแบบเดียวกัน แล้วส่งผล
# บรรทัดด้านล่าง (เมื่อลบ # ออก) จะเรียกฟังก์ชัน submit_to_leaderboard จากเซลล์การตั้งค่า
#   - "lab09_submission.json" : ชื่อไฟล์ผลลัพธ์ที่ต้องการส่งไปยังตารางอันดับ
# submit_to_leaderboard("lab09_submission.json")

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่ง**โน้ตบุ๊กนี้ที่รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (รันเซลล์ส่วนหลัก ✅ ครบถ้วน) พร้อมคำตอบของคำถาม ✍️ ทั้ง 5 ข้อ โน้ตบุ๊กควรแสดงข้อความที่ค้นคืนได้สำหรับคำถามของผู้เรียน การเปรียบเทียบคำตอบที่ไม่ใช้และใช้ RAG พร้อมผลการประเมิน เกณฑ์การงดตอบที่เลือก และการงดตอบคำถามนอกขอบเขตได้อย่างถูกต้อง โดยส่งตามช่องทางที่ระบุในเอกสารรายวิชา

ผู้เรียนที่ทำโจทย์ท้าทาย 🏆 บนตารางอันดับ สามารถระบุชื่อบนตารางอันดับและคะแนนสูงสุดไว้ในส่วนนี้ (ไม่บังคับ)

✍️ **ชื่อและคะแนนบนตารางอันดับ (ไม่บังคับ):**

## แหล่งศึกษาเพิ่มเติม
เครื่องมือสร้างระบบ RAG แบบไม่ต้องเขียนโปรแกรมหรือเขียนโปรแกรมเพียงเล็กน้อย (no-code/low-code) ได้แก่ **[Dify](https://dify.ai/)**, **[Langflow](https://www.langflow.org/)** และ **[Flowise](https://flowiseai.com/)** ซึ่งใช้แนวคิดเดียวกัน (embedding, vector store, การค้นคืน และ prompt) ผ่านการลากและวางองค์ประกอบ